# BeautyCore approved five-Hair + three-Nails release — rehearsal starter

Use a separate private copy of the tested pinned Kaggle notebook, reviewed T4 and Internet.
Attach exactly one capstone_train002_20260928_v1.bin and one render_kaggle_cpu_hair_nails_20261031_02.bin.
Enable the existing three Kaggle Secrets. Keep their values private; the backend key must match Render.
Run the three code cells once, in order, after matching Render code is deployed.
Five approved Hair styles use their exact reviewed TRAIN-001/TRAIN-002 prompts/settings.
Hair previews may change color/texture or cover the eyes; reviewed Butterfly may be bob-like, Wolf spiky.
The three tested Nail designs and existing Nails-only optimization are included.
New Makeup styles, Pearl Chrome and Cat-Eye remain disabled. Existing Makeup stays unchanged.
Original checkpoints, training metadata, registry files and original fallback notebooks/bundles are preserved.
Registration is automatic; no routine Render endpoint edit. Fresh GPU/website rehearsal is still required.


In [ ]:
"""Verified private bundles and early host preflight for the client demo notebook."""
from pathlib import Path
from hashlib import sha256
from zipfile import ZipFile
import json, platform, socket, sys, uuid

BUNDLES = (
    ("capstone_train002_20260928_v1.bin", "f0d1fed52e00b6c5a1da2cade3731237675cdebbfc0b2cde3b2dbda99a4019ac", "gpu-src", "gate3_bundle.json"),
    ('render_kaggle_cpu_hair_nails_20261031_02.bin', 'dfb5fc35f2c9c446fb8fdcc4439650c5f8a72941db864db1e9045cadae73082f', "cpu-src", "demo_bundle.json"),
)

def extract(source, expected, target, manifest_name):
    if sha256(source.read_bytes()).hexdigest() != expected:
        raise RuntimeError("Bundle hash differs: " + source.name)
    if target.exists():
        raise RuntimeError("Extraction requires a new directory; existing source is never overwritten")
    with ZipFile(source) as archive:
        manifest = json.loads(archive.read(manifest_name))
        names = archive.namelist()
        if len(names) != len(set(names)) or set(names) != set(manifest["files"]) | {manifest_name}:
            raise RuntimeError("Bundle member list differs")
        if archive.testzip() is not None:
            raise RuntimeError("Bundle archive integrity failed")
        # Validate every member before writing any of the archive contents.
        members = []
        for name in names:
            destination = (target / name).resolve()
            if ":" in name or chr(92) in name or not destination.is_relative_to(target.resolve()):
                raise RuntimeError("Unsafe bundle member path")
            raw = archive.read(name)
            if name != manifest_name and sha256(raw).hexdigest() != manifest["files"][name]:
                raise RuntimeError("Bundle member hash differs: " + name)
            members.append((destination, raw))
        for destination, raw in members:
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(raw)

def check_host(observed, expected):
    if any(observed.get(key) != expected.get(key) for key in ("python", "torch", "cuda", "gpu")):
        raise RuntimeError("Kaggle image differs from the reviewed GPU image. Expected "
                           + json.dumps(expected) + "; observed " + json.dumps(observed)
                           + ". Use the tested pinned notebook; do not change the model guards.")

def prepare(inputs=Path("/kaggle/input"), working=Path("/kaggle/working")):
    if sys.platform != "linux":
        raise RuntimeError("Run this starter inside Kaggle, not on the client device")
    for port in (8765, 8800):
        with socket.socket() as connection:
            connection.settimeout(1)
            if connection.connect_ex(("127.0.0.1", port)) == 0:
                raise RuntimeError("An AI service is already running. Do not run setup again.")
    work = working / ("render-kaggle-client-" + uuid.uuid4().hex)
    for name, digest, directory, manifest in BUNDLES:
        matches = list(inputs.rglob(name))
        if len(matches) != 1:
            raise RuntimeError("Attach exactly one input containing " + name)
        extract(matches[0], digest, work / directory, manifest)
    import torch
    observed = {"python": platform.python_version(), "torch": torch.__version__,
                "cuda": torch.version.cuda,
                "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
    review = json.loads((work / "gpu-src/docs/experiments/unified-kaggle-gate2-review.json").read_text())
    expected = {key: review["environment"][key] for key in observed}
    check_host(observed, expected)
    print("Private bundles and reviewed GPU host verified.", flush=True)
    print("Working directory:", work, flush=True)
    print("Run Cell 2 once. Keep this Kaggle session running during the demo.", flush=True)
    return work


def verify_release_pair(work, expected):
    gpu = json.loads((work / "gpu-src/gate3_bundle.json").read_text(encoding="utf-8"))
    cpu = json.loads((work / "cpu-src/demo_bundle.json").read_text(encoding="utf-8"))
    release = json.loads((work / "cpu-src/backend/app/style_release.json").read_text(encoding="utf-8"))
    canonical = json.dumps(release, sort_keys=True, ensure_ascii=True, separators=(",", ":")).encode()
    installed = sorted(c["id"] for c in release["candidates"] if c["enabled"])
    if (gpu.get("schema") != "unified-kaggle-gate3-v1"
            or cpu.get("schema") != "render-kaggle-cpu-v1"
            or cpu.get("gpu_bundle_sha256") != BUNDLES[0][1]
            or cpu.get("style_release") != expected
            or release.get("release_id") != expected["release_id"]
            or sha256(canonical).hexdigest() != expected["manifest_sha256"]
            or installed != expected["enabled_style_ids"]):
        raise RuntimeError("CPU/GPU style release pair differs; use the matching release notebook and inputs")
    return expected

STYLE_RELEASE_EXPECTED = {'release_id': 'styles-20261031-02', 'manifest_sha256': '6bbabcbc1a1364dc571036a404c7757f258d9fd2a1b9414ac39b4c9a10bf1db2', 'enabled_style_ids': ['exp_butterfly_cut', 'exp_comma_hair', 'exp_french_bob_wispy', 'exp_korean_two_block', 'exp_negative_space_microdots', 'exp_pink_aura', 'exp_raspberry_jelly', 'exp_wolf_cut']}
CLIENT_WORK = prepare()
verify_release_pair(CLIENT_WORK, STYLE_RELEASE_EXPECTED)
"""Stage one matched Hair/Nails worker release in a fresh verified GPU workspace."""
import base64
from hashlib import sha256
import json
from pathlib import Path
import socket

FILES = frozenset({
    "scripts/unified_gate3_server.py", "scripts/inference_performance.py", "scripts/hair_release_worker.py",
    "backend/app/hair_profiles.py", "backend/app/hair_profile.json",
    "backend/app/hair_profile_butterfly_cut.json", "backend/app/hair_profile_wolf_cut.json",
    "backend/app/hair_profile_korean_two_block.json", "backend/app/hair_profile_comma_hair.json",
    "backend/app/style_release.py", "backend/app/style_release.json",
})


def validated(payload):
    if not isinstance(payload, dict) or set(payload) != FILES:
        raise ValueError("Unexpected Hair/Nails worker source inventory")
    sources = {}
    for name, item in payload.items():
        if not isinstance(item, dict) or set(item) != {"data", "sha256"}:
            raise ValueError("Invalid release source record")
        raw = base64.b64decode(item["data"], validate=True)
        if sha256(raw).hexdigest() != item["sha256"]:
            raise ValueError("Release source hash differs: " + name)
        if name.endswith(".py"):
            compile(raw, name, "exec")
        else:
            json.loads(raw)
        sources[name] = raw
    return sources


def stage_overlay(work, payload):
    validated(payload)
    target = Path(work) / "optimized-worker.json"
    with target.open("x", encoding="utf-8") as file:
        json.dump(payload, file)
    return target


def install_overlay(root, payload_file, output):
    root, output = Path(root), Path(output)
    for port in (8765, 8800):
        with socket.socket() as connection:
            connection.settimeout(1)
            if connection.connect_ex(("127.0.0.1", port)) == 0:
                raise RuntimeError("Do not change a running AI service")
    record = output / "hair-nails-worker-release.json"
    if record.exists():
        raise RuntimeError("Worker release already installed; use a fresh session")
    sources = validated(json.loads(Path(payload_file).read_text(encoding="utf-8")))
    manifest_path = root / "gate3_bundle.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    # Full original bundle verification is also required by the startup hook.
    # Never rewrite the original manifest or any model/registry/checkpoint.
    for name in FILES:
        target = root / name
        if name in manifest["files"]:
            if sha256(target.read_bytes()).hexdigest() != manifest["files"][name]:
                raise ValueError("Original worker source changed before release install")
        elif target.exists():
            raise ValueError("Unexpected preexisting worker extension: " + name)
    for name, raw in sources.items():
        target = root / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(raw)
    evidence = {"schema": "beautycore-hair-nails-worker-v1",
                "original_manifest_sha256": sha256(manifest_path.read_bytes()).hexdigest(),
                "source_sha256": {name: sha256(raw).hexdigest() for name, raw in sources.items()},
                "model_parameters_changed": False,
                "hair_profile": "hair-expanded-v1", "optimization": "nails-only"}
    record.write_bytes((json.dumps(evidence, indent=2) + "\n").encode())
    print("WORKER RELEASE: approved Hair profiles and existing Nails-only optimization.", flush=True)
    return evidence

OPTIMIZED_SOURCES = {'backend/app/hair_profile.json': {'data': 'ewogICJzY2hlbWEiOiAiYmVhdXR5Y29yZS1oYWlyLXByb2ZpbGUtdjEiLAogICJwcm9maWxlX2lkIjogImhhaXItZnJlbmNoLWJvYi12MSIsCiAgInN0eWxlX2lkIjogImV4cF9mcmVuY2hfYm9iX3dpc3B5IiwKICAiYWRhcHRlcl9pZCI6ICJ0cmFpbjAwMSIsCiAgImFkYXB0ZXJfc2hhMjU2IjogIjdlMzk5MWY4YTRlNTAyNTczZDNlODJlOWFjMzRjODlmZGYzZjBiNjZmYjMzN2FiYjAyNmE1YjRjOWFkMDgwZmYiLAogICJiYXNlX21vZGVsX3JldmlzaW9uIjogImEzYjRmNDg0OTE1N2Y2NjRiZGJjNzc2ZmQ3NDUzYzI3ODM1NjJmNGQiLAogICJwcm9tcHQiOiAiQ2hhbmdlIHRoZSBoYWlyc3R5bGUgdG8gYSB0ZXh0dXJlZCBGcmVuY2ggYm9iIGVuZGluZyBhcm91bmQgdGhlIGphdywgd2l0aCBhIHNvZnRseSBicm9rZW4gb3V0bGluZSBhbmQgdGhpbiwgYWlyeSwgc2VwYXJhdGVkIHdpc3B5IGJhbmdzIGFib3ZlIHRoZSBleWVzLiBLZWVwIHRoZSBmcmluZ2UgbGlnaHQgcmF0aGVyIHRoYW4gYSBoZWF2eSBibHVudCBjdXJ0YWluLiBLZWVwIHRoZSBzYW1lIHBlcnNvbiwgcmVjb2duaXphYmxlIGZhY2lhbCBwcm9wb3J0aW9ucywgZXhwcmVzc2lvbiwgbmF0dXJhbCBza2luIHRvbmUsIG9yaWdpbmFsIGhhaXIgY29sb3IsIGNsb3RoaW5nLCBwb3NlLCBsaWdodGluZywgZnJhbWluZyBhbmQgYmFja2dyb3VuZC4gQ2hhbmdlIGhhaXJzdHlsZSBvbmx5LiIsCiAgInNldHRpbmdzIjogewogICAgIndpZHRoIjogNTEyLAogICAgImhlaWdodCI6IDUxMiwKICAgICJzdGVwcyI6IDIwLAogICAgImd1aWRhbmNlIjogNC4wLAogICAgInNlZWQiOiAxOTc3LAogICAgImFkYXB0ZXJfd2VpZ2h0IjogMS4wCiAgfSwKICAibGltaXRhdGlvbnMiOiBbCiAgICAiUHJldmlldyBtYXkgY2hhbmdlIGhhaXIgY29sb3IgYW5kIHRleHR1cmU7IHRoZSBhcHByb3ZlZCBleGFtcGxlcyBzaG93IGEgY3VybHkgaW50ZXJwcmV0YXRpb24uIgogIF0sCiAgInZhbGlkYXRpb24iOiB7CiAgICAiZXhwZXJpbWVudCI6ICJIQUlSLUVYUC0wMDIiLAogICAgInJlc3VsdHNfc2hhMjU2IjogIjNlMzQ3NGI5ODgyMzI0N2I0NzE5NjRkZTE3MzIyM2FlNmViMzE0YWYzNGNkYWFmYzQ4NjAxMmVlODhjYWUxZGIiLAogICAgImNhc2VzIjogWwogICAgICA2LAogICAgICA4CiAgICBdCiAgfQp9Cg==', 'sha256': 'fec43c31a04e1cea7b18766026f088d63534429ddfc6aed9d2723b4a86a18d0d'}, 'backend/app/hair_profile_butterfly_cut.json': {'data': 'ewogICJzY2hlbWEiOiAiYmVhdXR5Y29yZS1oYWlyLXByb2ZpbGUtdjEiLAogICJwcm9maWxlX2lkIjogImhhaXItYnV0dGVyZmx5LWN1dC12MSIsCiAgInN0eWxlX2lkIjogImV4cF9idXR0ZXJmbHlfY3V0IiwKICAiYWRhcHRlcl9pZCI6ICJ0cmFpbjAwMSIsCiAgImFkYXB0ZXJfc2hhMjU2IjogIjdlMzk5MWY4YTRlNTAyNTczZDNlODJlOWFjMzRjODlmZGYzZjBiNjZmYjMzN2FiYjAyNmE1YjRjOWFkMDgwZmYiLAogICJiYXNlX21vZGVsX3JldmlzaW9uIjogImEzYjRmNDg0OTE1N2Y2NjRiZGJjNzc2ZmQ3NDUzYzI3ODM1NjJmNGQiLAogICJwcm9tcHQiOiAiQ2hhbmdlIHRoZSBoYWlyc3R5bGUgdG8gYSBidXR0ZXJmbHkgY3V0OiByZXRhaW4gbG9uZyBsb3dlciBsZW5ndGhzLCBjcmVhdGUgc2hvcnRlciBwcm9ub3VuY2VkIGZhY2UtZnJhbWluZyB1cHBlciBsYXllcnMgYW5kIHZvbHVtaW5vdXMgY3VydmVkIGxheWVycyBzd2VlcGluZyBhd2F5IGZyb20gdGhlIGZhY2UuIEtlZXAgdGhlIHNhbWUgcGVyc29uLCByZWNvZ25pemFibGUgZmFjZSwgZmFjaWFsIHByb3BvcnRpb25zLCBleHByZXNzaW9uLCBza2luIHRvbmUsIGNsb3RoaW5nLCBwb3NlLCBiYWNrZ3JvdW5kLCBsaWdodGluZyBhbmQgZnJhbWluZy4gUHJlc2VydmUgdGhlIG9yaWdpbmFsIGhhaXIgY29sb3IuIENoYW5nZSBoYWlyc3R5bGUgb25seS4iLAogICJzZXR0aW5ncyI6IHsKICAgICJ3aWR0aCI6IDUxMiwKICAgICJoZWlnaHQiOiA1MTIsCiAgICAic3RlcHMiOiAyMCwKICAgICJndWlkYW5jZSI6IDQuMCwKICAgICJzZWVkIjogMTk3NywKICAgICJhZGFwdGVyX3dlaWdodCI6IDEuMAogIH0sCiAgImxpbWl0YXRpb25zIjogWwogICAgIkJ1dHRlcmZseS1pbnNwaXJlZCBsYXllcnM7IHJldmlld2VkIHByZXZpZXdzIGNhbiBhcHBlYXIgc2hvcnRlciBhbmQgYm9iLWxpa2UuIFByZXZpZXcgbWF5IGNoYW5nZSBoYWlyIGNvbG9yLiIKICBdLAogICJ2YWxpZGF0aW9uIjogewogICAgImV4cGVyaW1lbnQiOiAiSEFJUi1FWFAtMDAyIiwKICAgICJyZXN1bHRzX3NoYTI1NiI6ICIzZTM0NzRiOTg4MjMyNDdiNDcxOTY0ZGUxNzMyMjNhZTZlYjMxNGFmMzRjZGFhZmM0ODYwMTJlZTg4Y2FlMWRiIiwKICAgICJzdWJqZWN0X2lkcyI6IFsKICAgICAgImQyX3dhdnlfaGFpcl8yNWpwZyIsCiAgICAgICJkMl93YXZ5X2hhaXJfMjdqcGciCiAgICBdCiAgfQp9Cg==', 'sha256': '6a63f309460dfb1bc5cbf0fc7e3584fa66fd20f560abbcedf5dcb25cd8338770'}, 'backend/app/hair_profile_comma_hair.json': {'data': 'ewogICJzY2hlbWEiOiAiYmVhdXR5Y29yZS1oYWlyLXByb2ZpbGUtdjEiLAogICJwcm9maWxlX2lkIjogImhhaXItY29tbWEtaGFpci12MSIsCiAgInN0eWxlX2lkIjogImV4cF9jb21tYV9oYWlyIiwKICAiYWRhcHRlcl9pZCI6ICJ0cmFpbjAwMiIsCiAgImFkYXB0ZXJfc2hhMjU2IjogIjU5ZDUyNzIxN2E2ODEzOWJjMmFhYTRlZWRiZmNkMmQ1Nzg5ZWYxMTVmOGVjYjkyNDQxMGVlYTI0YTk1MjkzOWIiLAogICJiYXNlX21vZGVsX3JldmlzaW9uIjogImEzYjRmNDg0OTE1N2Y2NjRiZGJjNzc2ZmQ3NDUzYzI3ODM1NjJmNGQiLAogICJwcm9tcHQiOiAiU3R5bGUgdGhlIGhhaXIgaW50byBLb3JlYW4gY29tbWEgaGFpciB3aXRoIGEgc2lkZSBwYXJ0IGFuZCBhIHNvZnRseSBzd2VwdCBmcmluZ2UgY2xlYXIgb2YgdGhlIGV5ZXMuIEN1cnZlIHRoZSBlbmRzIG9mIGEgYnJvYWQgZnJvbnQgZnJpbmdlIGlud2FyZCBhcyBuYXR1cmFsIGluZGl2aWR1YWwgaGFpciBzdHJhbmRzIGludG8gYSBzb2Z0IEMgc2hhcGUgYWJvdmUgb25lIGJyb3csIGJsZW5kaW5nIGludG8gdGhlIHRvcC4gQXZvaWQgYSBkZXRhY2hlZCBjdXJsZWQgbG9jaywgY2xvc2VkIHJpbmcgb3IgcGFpbnRlZCBsb29wLiBLZWVwIHRoZSBzYW1lIHBlcnNvbiwgcmVjb2duaXphYmxlIGZhY2lhbCBwcm9wb3J0aW9ucywgZXhwcmVzc2lvbiwgbmF0dXJhbCBza2luIHRvbmUsIG9yaWdpbmFsIGhhaXIgY29sb3IsIGNsb3RoaW5nLCBwb3NlLCBsaWdodGluZywgZnJhbWluZyBhbmQgYmFja2dyb3VuZC4gQ2hhbmdlIGhhaXJzdHlsZSBvbmx5LiIsCiAgInNldHRpbmdzIjogewogICAgIndpZHRoIjogNTEyLAogICAgImhlaWdodCI6IDUxMiwKICAgICJzdGVwcyI6IDIwLAogICAgImd1aWRhbmNlIjogNC4wLAogICAgInNlZWQiOiAxOTc3LAogICAgImFkYXB0ZXJfd2VpZ2h0IjogMS4wCiAgfSwKICAibGltaXRhdGlvbnMiOiBbCiAgICAiQSBzaWRlLXBhcnRlZCwgaW53YXJkLWN1cnZlZCBmcmluZ2UgaW5zcGlyZWQgYnkgS29yZWFuIGNvbW1hIGhhaXIuIFByZXZpZXcgbWF5IHJlc2VtYmxlIGN1cnRhaW4gaGFpciwgY292ZXIgZXllcyBvciBjaGFuZ2UgaGFpciBjb2xvci4iCiAgXSwKICAidmFsaWRhdGlvbiI6IHsKICAgICJleHBlcmltZW50IjogIkhBSVItRVhQLTAwMiIsCiAgICAicmVzdWx0c19zaGEyNTYiOiAiM2UzNDc0Yjk4ODIzMjQ3YjQ3MTk2NGRlMTczMjIzYWU2ZWIzMTRhZjM0Y2RhYWZjNDg2MDEyZWU4OGNhZTFkYiIsCiAgICAic3ViamVjdF9pZHMiOiBbCiAgICAgICJkMl9zaWRlX3BhcnRfdW5kZXJjdXRfMjhqcGciLAogICAgICAiZDJfc2lkZV9wYXJ0X3VuZGVyY3V0XzlqcGciCiAgICBdCiAgfQp9Cg==', 'sha256': 'd7cc5ab8870cfc9d014c65c2b5d9c58f5ce5a5e87f78511636057c3620887668'}, 'backend/app/hair_profile_korean_two_block.json': {'data': 'ewogICJzY2hlbWEiOiAiYmVhdXR5Y29yZS1oYWlyLXByb2ZpbGUtdjEiLAogICJwcm9maWxlX2lkIjogImhhaXIta29yZWFuLXR3by1ibG9jay12MSIsCiAgInN0eWxlX2lkIjogImV4cF9rb3JlYW5fdHdvX2Jsb2NrIiwKICAiYWRhcHRlcl9pZCI6ICJ0cmFpbjAwMiIsCiAgImFkYXB0ZXJfc2hhMjU2IjogIjU5ZDUyNzIxN2E2ODEzOWJjMmFhYTRlZWRiZmNkMmQ1Nzg5ZWYxMTVmOGVjYjkyNDQxMGVlYTI0YTk1MjkzOWIiLAogICJiYXNlX21vZGVsX3JldmlzaW9uIjogImEzYjRmNDg0OTE1N2Y2NjRiZGJjNzc2ZmQ3NDUzYzI3ODM1NjJmNGQiLAogICJwcm9tcHQiOiAiQ2hhbmdlIHRoZSBoYWlyc3R5bGUgdG8gYSBLb3JlYW4gdHdvLWJsb2NrIGhhaXJjdXQgd2l0aCBhIHZpc2libHkgc2hvcnRlciBsb3dlciBzZWN0aW9uIGF0IHRoZSBzaWRlcyBhbmQgbmFwZSBhbmQgYSBkaXN0aW5jdGx5IGxvbmdlciBmdWxsZXIgdXBwZXIgc2VjdGlvbiB3aXRoIGEgc29mdCBuYXR1cmFsIGZyaW5nZS4gU2hvdyB0aGUgdHdvIGxlbmd0aHMgY2xlYXJseS4gS2VlcCB0aGUgc2FtZSBwZXJzb24sIHJlY29nbml6YWJsZSBmYWNlLCBmYWNpYWwgcHJvcG9ydGlvbnMsIGV4cHJlc3Npb24sIHNraW4gdG9uZSwgY2xvdGhpbmcsIHBvc2UsIGJhY2tncm91bmQsIGxpZ2h0aW5nIGFuZCBmcmFtaW5nLiBQcmVzZXJ2ZSB0aGUgb3JpZ2luYWwgaGFpciBjb2xvci4gQ2hhbmdlIGhhaXJzdHlsZSBvbmx5LiIsCiAgInNldHRpbmdzIjogewogICAgIndpZHRoIjogNTEyLAogICAgImhlaWdodCI6IDUxMiwKICAgICJzdGVwcyI6IDIwLAogICAgImd1aWRhbmNlIjogNC4wLAogICAgInNlZWQiOiAxOTc3LAogICAgImFkYXB0ZXJfd2VpZ2h0IjogMS4wCiAgfSwKICAibGltaXRhdGlvbnMiOiBbCiAgICAiQSBmdWxsIHVwcGVyIHNlY3Rpb24gYW5kIGRlbnNlIGZyaW5nZSBpbnNwaXJlZCBieSB0aGUgS29yZWFuIHR3by1ibG9jayBjdXQuIFByZXZpZXcgbWF5IGNvdmVyIGV5ZXMgb3IgY2hhbmdlIGhhaXIgY29sb3IuIgogIF0sCiAgInZhbGlkYXRpb24iOiB7CiAgICAiZXhwZXJpbWVudCI6ICJIQUlSLUVYUC0wMDIiLAogICAgInJlc3VsdHNfc2hhMjU2IjogIjNlMzQ3NGI5ODgyMzI0N2I0NzE5NjRkZTE3MzIyM2FlNmViMzE0YWYzNGNkYWFmYzQ4NjAxMmVlODhjYWUxZGIiLAogICAgInN1YmplY3RfaWRzIjogWwogICAgICAiZDJfc2lkZV9wYXJ0X3VuZGVyY3V0XzI4anBnIiwKICAgICAgImQyX3NpZGVfcGFydF91bmRlcmN1dF85anBnIgogICAgXQogIH0KfQo=', 'sha256': '0913d54db408fa79e9084a208c4049afb58d9651091107398371e53f82bc10c1'}, 'backend/app/hair_profile_wolf_cut.json': {'data': 'ewogICJzY2hlbWEiOiAiYmVhdXR5Y29yZS1oYWlyLXByb2ZpbGUtdjEiLAogICJwcm9maWxlX2lkIjogImhhaXItd29sZi1jdXQtdjEiLAogICJzdHlsZV9pZCI6ICJleHBfd29sZl9jdXQiLAogICJhZGFwdGVyX2lkIjogInRyYWluMDAyIiwKICAiYWRhcHRlcl9zaGEyNTYiOiAiNTlkNTI3MjE3YTY4MTM5YmMyYWFhNGVlZGJmY2QyZDU3ODllZjExNWY4ZWNiOTI0NDEwZWVhMjRhOTUyOTM5YiIsCiAgImJhc2VfbW9kZWxfcmV2aXNpb24iOiAiYTNiNGY0ODQ5MTU3ZjY2NGJkYmM3NzZmZDc0NTNjMjc4MzU2MmY0ZCIsCiAgInByb21wdCI6ICJDaGFuZ2UgdGhlIGhhaXJzdHlsZSB0byBhIHdvbGYgY3V0IHdpdGggYSBzaG9ydCwgY2hvcHB5LCB2b2x1bWlub3VzIGNyb3duLCBmZWF0aGVyZWQgZmFjZS1mcmFtaW5nIGxheWVycyBhbmQgYSBkaXN0aW5jdGx5IGxvbmdlciwgd2lzcHkgYmFjay4gS2VlcCBhIHZpc2libGUgY29udHJhc3QgYmV0d2VlbiB0aGUgc2hvcnRlciB1cHBlciBsYXllcnMgYW5kIGxvbmdlciBsb3dlciBsZW5ndGhzLCByYXRoZXIgdGhhbiBhIHVuaWZvcm0gc2hhZy4gS2VlcCB0aGUgc2FtZSBwZXJzb24sIHJlY29nbml6YWJsZSBmYWNpYWwgcHJvcG9ydGlvbnMsIGV4cHJlc3Npb24sIG5hdHVyYWwgc2tpbiB0b25lLCBvcmlnaW5hbCBoYWlyIGNvbG9yLCBjbG90aGluZywgcG9zZSwgbGlnaHRpbmcsIGZyYW1pbmcgYW5kIGJhY2tncm91bmQuIENoYW5nZSBoYWlyc3R5bGUgb25seS4iLAogICJzZXR0aW5ncyI6IHsKICAgICJ3aWR0aCI6IDUxMiwKICAgICJoZWlnaHQiOiA1MTIsCiAgICAic3RlcHMiOiAyMCwKICAgICJndWlkYW5jZSI6IDQuMCwKICAgICJzZWVkIjogMTk3NywKICAgICJhZGFwdGVyX3dlaWdodCI6IDEuMAogIH0sCiAgImxpbWl0YXRpb25zIjogWwogICAgIkEgYm9sZCwgc3Bpa3kgaW50ZXJwcmV0YXRpb24gb2YgYSBjaG9wcHkgd29sZiBjdXQgd2l0aCBhIGxvbmdlciBiYWNrLiBQcmV2aWV3IG1heSBjaGFuZ2UgaGFpciBjb2xvci4iCiAgXSwKICAidmFsaWRhdGlvbiI6IHsKICAgICJleHBlcmltZW50IjogIkhBSVItRVhQLTAwMiIsCiAgICAicmVzdWx0c19zaGEyNTYiOiAiM2UzNDc0Yjk4ODIzMjQ3YjQ3MTk2NGRlMTczMjIzYWU2ZWIzMTRhZjM0Y2RhYWZjNDg2MDEyZWU4OGNhZTFkYiIsCiAgICAic3ViamVjdF9pZHMiOiBbCiAgICAgICJkMl93YXZ5X2hhaXJfMjVqcGciLAogICAgICAiZDJfd2F2eV9oYWlyXzI3anBnIgogICAgXQogIH0KfQo=', 'sha256': 'a79f39cc7d3e917a390d9dfc62de65d69185de8a68c4dda09e21048c9d32d123'}, 'backend/app/hair_profiles.py': {'data': 'IiIiQXBwcm92ZWQgSGFpciBpbmZlcmVuY2UgcHJvZmlsZXM7IGZyb3plbiB0cmFpbmluZyByZWdpc3RyaWVzIHN0YXkgaW50YWN0LiIiIgppbXBvcnQganNvbgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSBhcHAuc3R5bGVfcmVsZWFzZSBpbXBvcnQgcHJvZmlsZV9kaWdlc3QKClBBVEggPSBQYXRoKF9fZmlsZV9fKS53aXRoX25hbWUoImhhaXJfcHJvZmlsZS5qc29uIikKUFJPRklMRSA9IGpzb24ubG9hZHMoUEFUSC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpClNUWUxFX0lEID0gImV4cF9mcmVuY2hfYm9iX3dpc3B5IgpQUk9GSUxFX0lEID0gImhhaXItZnJlbmNoLWJvYi12MSIKV09SS0VSX1NDSEVNQSA9ICJiZWF1dHljb3JlLXdvcmtlci1zdHlsZS1jYXBhYmlsaXRpZXMtdjEiCkVYVFJBID0gewogICAgImV4cF9idXR0ZXJmbHlfY3V0IjogKCJoYWlyLWJ1dHRlcmZseS1jdXQtdjEiLCAidHJhaW4wMDEiKSwKICAgICJleHBfd29sZl9jdXQiOiAoImhhaXItd29sZi1jdXQtdjEiLCAidHJhaW4wMDIiKSwKICAgICJleHBfa29yZWFuX3R3b19ibG9jayI6ICgiaGFpci1rb3JlYW4tdHdvLWJsb2NrLXYxIiwgInRyYWluMDAyIiksCiAgICAiZXhwX2NvbW1hX2hhaXIiOiAoImhhaXItY29tbWEtaGFpci12MSIsICJ0cmFpbjAwMiIpLAp9ClNUWUxFX0lEUyA9IGZyb3plbnNldCh7U1RZTEVfSUQsICpFWFRSQX0pCkFEQVBURVJfSEFTSEVTID0gewogICAgInRyYWluMDAxIjogIjdlMzk5MWY4YTRlNTAyNTczZDNlODJlOWFjMzRjODlmZGYzZjBiNjZmYjMzN2FiYjAyNmE1YjRjOWFkMDgwZmYiLAogICAgInRyYWluMDAyIjogIjU5ZDUyNzIxN2E2ODEzOWJjMmFhYTRlZWRiZmNkMmQ1Nzg5ZWYxMTVmOGVjYjkyNDQxMGVlYTI0YTk1MjkzOWIiLAp9CgoKZGVmIHByb2ZpbGUocmVsZWFzZT1Ob25lLCByZWdpc3RyeT1Ob25lLCBzdHlsZV9pZD1TVFlMRV9JRCk6CiAgICBmcm9tIGFwcC5zdHlsZV9yZWxlYXNlIGltcG9ydCBSRUxFQVNFCiAgICBmcm9tIGFwcC5zdHlsZXMgaW1wb3J0IFJFR0lTVFJZCiAgICByZWxlYXNlID0gUkVMRUFTRSBpZiByZWxlYXNlIGlzIE5vbmUgZWxzZSByZWxlYXNlCiAgICByZWdpc3RyeSA9IFJFR0lTVFJZIGlmIHJlZ2lzdHJ5IGlzIE5vbmUgZWxzZSByZWdpc3RyeQogICAgYyA9IG5leHQoKGMgZm9yIGMgaW4gcmVsZWFzZS5jYW5kaWRhdGVzIGlmIGMuaWQgPT0gc3R5bGVfaWQpLCBOb25lKQogICAgaWYgYyBpcyBOb25lIG9yIGMucHJvZmlsZV9pZCBpcyBOb25lOgogICAgICAgIHJldHVybiBOb25lCiAgICBpZiBzdHlsZV9pZCA9PSBTVFlMRV9JRDoKICAgICAgICBwYXRoLCBwLCBwcm9maWxlX2lkLCBhZGFwdGVyX2lkID0gUEFUSCwgUFJPRklMRSwgUFJPRklMRV9JRCwgInRyYWluMDAxIgogICAgZWxpZiBzdHlsZV9pZCBpbiBFWFRSQToKICAgICAgICBwcm9maWxlX2lkLCBhZGFwdGVyX2lkID0gRVhUUkFbc3R5bGVfaWRdCiAgICAgICAgcGF0aCA9IFBhdGgoX19maWxlX18pLndpdGhfbmFtZSgiaGFpcl9wcm9maWxlXyIgKyBzdHlsZV9pZC5yZW1vdmVwcmVmaXgoImV4cF8iKSArICIuanNvbiIpCiAgICAgICAgcCA9IGpzb24ubG9hZHMocGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBlbHNlOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlN0eWxlIGhhcyBubyBtYXRjaGluZyBpbnN0YWxsZWQgSGFpciBleGVjdXRpb24gcHJvZmlsZTogIiArIHN0eWxlX2lkKQogICAgaWYgKGMuZmVhdHVyZSAhPSAiaGFpcnN0eWxlIiBvciBjLnJvdXRlICE9ICJoYWlyX3Byb21wdCIgb3IgYy5wcm9maWxlX2lkICE9IHByb2ZpbGVfaWQKICAgICAgICAgICAgb3IgYy5wcm9maWxlX3NoYTI1NiAhPSBwcm9maWxlX2RpZ2VzdChwYXRoKQogICAgICAgICAgICBvciBzZXQocCkgIT0geyJzY2hlbWEiLCAicHJvZmlsZV9pZCIsICJzdHlsZV9pZCIsICJhZGFwdGVyX2lkIiwgImFkYXB0ZXJfc2hhMjU2IiwKICAgICAgICAgICAgICAgICAgICAgICAgICJiYXNlX21vZGVsX3JldmlzaW9uIiwgInByb21wdCIsICJzZXR0aW5ncyIsICJsaW1pdGF0aW9ucyIsICJ2YWxpZGF0aW9uIn0KICAgICAgICAgICAgb3IgcFsic2NoZW1hIl0gIT0gImJlYXV0eWNvcmUtaGFpci1wcm9maWxlLXYxIiBvciBwWyJwcm9maWxlX2lkIl0gIT0gcHJvZmlsZV9pZAogICAgICAgICAgICBvciBwWyJzdHlsZV9pZCJdICE9IHN0eWxlX2lkIG9yIHBbImFkYXB0ZXJfaWQiXSAhPSBhZGFwdGVyX2lkCiAgICAgICAgICAgIG9yIHBbImFkYXB0ZXJfc2hhMjU2Il0gIT0gQURBUFRFUl9IQVNIRVNbYWRhcHRlcl9pZF0KICAgICAgICAgICAgb3IgcFsic2V0dGluZ3MiXSAhPSB7IndpZHRoIjogNTEyLCAiaGVpZ2h0IjogNTEyLCAic3RlcHMiOiAyMCwgImd1aWRhbmNlIjogNC4wLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJzZWVkIjogMTk3NywgImFkYXB0ZXJfd2VpZ2h0IjogMS4wfQogICAgICAgICAgICBvciBub3QgaXNpbnN0YW5jZShwWyJwcm9tcHQiXSwgc3RyKSBvciBub3QgMjAgPD0gbGVuKHBbInByb21wdCJdKSA8PSAyMDAwCiAgICAgICAgICAgIG9yIHJlZ2lzdHJ5WyJiYXNlX21vZGVsX3JldmlzaW9uIl0gIT0gcFsiYmFzZV9tb2RlbF9yZXZpc2lvbiJdCiAgICAgICAgICAgIG9yIGFueShzWyJzdHlsZV9pZCJdID09IHN0eWxlX2lkIGZvciBzIGluIHJlZ2lzdHJ5WyJzdHlsZXMiXSkpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkFwcHJvdmVkIEhhaXIgaW5mZXJlbmNlIHByb2ZpbGUgZGlmZmVycyBmcm9tIGl0cyByZWxlYXNlL2FydGlmYWN0IikKICAgICMgQSByZWdpc3RyeSB3aXRob3V0IFRSQUlOLTAwMiBtdXN0IG5ldmVyIGFkdmVydGlzZSBpdHMgYWRkaXRpb25hbCBzdHlsZXMuCiAgICBpbmZvID0gcmVnaXN0cnlbImFkYXB0ZXJzIl0uZ2V0KGFkYXB0ZXJfaWQpCiAgICBpZiBpbmZvIGlzIE5vbmUgb3IgaW5mb1sic3RhdHVzIl0gIT0gImVuYWJsZWQiOgogICAgICAgIHJldHVybiBOb25lCiAgICBpZiBpbmZvWyJjaGVja3BvaW50X3NoYTI1NiJdICE9IHBbImFkYXB0ZXJfc2hhMjU2Il06CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQXBwcm92ZWQgSGFpciBpbmZlcmVuY2UgcHJvZmlsZSBkaWZmZXJzIGZyb20gaXRzIHJlbGVhc2UvYXJ0aWZhY3QiKQogICAgcmV0dXJuIGMsIHAKCgpkZWYgaGFpcl9jYW5kaWRhdGVzKHJlbGVhc2U9Tm9uZSwgcmVnaXN0cnk9Tm9uZSk6CiAgICBmcm9tIGFwcC5zdHlsZV9yZWxlYXNlIGltcG9ydCBSRUxFQVNFCiAgICByZWxlYXNlID0gUkVMRUFTRSBpZiByZWxlYXNlIGlzIE5vbmUgZWxzZSByZWxlYXNlCiAgICByZXN1bHQgPSBbXQogICAgZm9yIGMgaW4gcmVsZWFzZS5jYW5kaWRhdGVzOgogICAgICAgIGlmIGMuZmVhdHVyZSAhPSAiaGFpcnN0eWxlIiBvciBjLnByb2ZpbGVfaWQgaXMgTm9uZToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBjLmlkIG5vdCBpbiBTVFlMRV9JRFM6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlN0eWxlIGhhcyBubyBtYXRjaGluZyBpbnN0YWxsZWQgSGFpciBleGVjdXRpb24gcHJvZmlsZTogIiArIGMuaWQpCiAgICAgICAgdmVyaWZpZWQgPSBwcm9maWxlKHJlbGVhc2UsIHJlZ2lzdHJ5LCBjLmlkKQogICAgICAgIGlmIHZlcmlmaWVkIGlzIG5vdCBOb25lIGFuZCB2ZXJpZmllZFswXS5lbmFibGVkOgogICAgICAgICAgICByZXN1bHQuYXBwZW5kKHZlcmlmaWVkWzBdKQogICAgcmV0dXJuIHR1cGxlKHJlc3VsdCkKCgpkZWYgYWN0aXZlX2hhaXJfc3R5bGVzKHJlbGVhc2U9Tm9uZSk6CiAgICBmcm9tIGFwcC5zdHlsZXMgaW1wb3J0IFJFQUxfU1RZTEVTLCBTdHlsZQogICAgcmV0dXJuIFJFQUxfU1RZTEVTICsgdHVwbGUoU3R5bGUoYy5pZCwgYy5uYW1lLCBjLmRlc2NyaXB0aW9uLCAiYXZhaWxhYmxlIikgZm9yIGMgaW4gaGFpcl9jYW5kaWRhdGVzKHJlbGVhc2UpKQoKCmRlZiBhY3RpdmVfaGFpcl9zdHlsZV9ieV9pZChyZWxlYXNlPU5vbmUpOgogICAgcmV0dXJuIHtzLmlkOiBzIGZvciBzIGluIGFjdGl2ZV9oYWlyX3N0eWxlcyhyZWxlYXNlKX0KCgpkZWYgbW9kZWxfcHJvZmlsZShyZWxlYXNlPU5vbmUpOgogICAgYWRkaXRpb25zID0gaGFpcl9jYW5kaWRhdGVzKHJlbGVhc2UpCiAgICBpZiBhbnkoYy5pZCAhPSBTVFlMRV9JRCBmb3IgYyBpbiBhZGRpdGlvbnMpOgogICAgICAgIHJldHVybiAiaGFpci1leHBhbmRlZC12MSIKICAgIHJldHVybiBQUk9GSUxFX0lEIGlmIGFkZGl0aW9ucyBlbHNlICJsZWdhY3ktYmFzZWxpbmUiCgoKZGVmIGV4cGVjdGVkX2NvbnRyYWN0KHN0eWxlX2lkLCBtZXRhZGF0YT1Ob25lKToKICAgIGZyb20gaGFzaGxpYiBpbXBvcnQgc2hhMjU2CiAgICBmcm9tIGFwcC5zdHlsZV9yZWxlYXNlIGltcG9ydCBSRUxFQVNFCiAgICBmcm9tIGFwcC5zdHlsZXMgaW1wb3J0IFJFR0lTVFJZCiAgICBpZiBzdHlsZV9pZCBub3QgaW4ge2MuaWQgZm9yIGMgaW4gaGFpcl9jYW5kaWRhdGVzKCl9OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkFkZGl0aW9uYWwgSGFpciBzdHlsZSBpcyBub3QgYXBwcm92ZWQvZW5hYmxlZCIpCiAgICBjLCBwID0gcHJvZmlsZShzdHlsZV9pZD1zdHlsZV9pZCkKICAgIGluZm8gPSBSRUdJU1RSWVsiYWRhcHRlcnMiXVtwWyJhZGFwdGVyX2lkIl1dCiAgICBpZiBtZXRhZGF0YSBpcyBub3QgTm9uZSBhbmQgKAogICAgICAgICAgICBtZXRhZGF0YS5nZXQoImNoZWNrcG9pbnRfc2hhMjU2IikgIT0gcFsiYWRhcHRlcl9zaGEyNTYiXQogICAgICAgICAgICBvciBtZXRhZGF0YS5nZXQoInRyYWluaW5nX3N0ZXBzIikgIT0gaW5mb1sidHJhaW5pbmdfc3RlcHMiXQogICAgICAgICAgICBvciBtZXRhZGF0YS5nZXQoImJhc2VfbW9kZWxfcmV2aXNpb24iKSAhPSBwWyJiYXNlX21vZGVsX3JldmlzaW9uIl0KICAgICAgICAgICAgb3IgbWV0YWRhdGEuZ2V0KCJiYXNlX21vZGVsX2lkIikgIT0gUkVHSVNUUllbImJhc2VfbW9kZWxfaWQiXQogICAgICAgICAgICBvciBzdHlsZV9pZCBpbiBtZXRhZGF0YS5nZXQoInN1cHBvcnRlZF9zdHlsZV9pZHMiLCBbXSkpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkhhaXIgYXJ0aWZhY3QgbWV0YWRhdGEgb3Igb3JpZ2luYWwgdHJhaW5pbmcgY2xhaW1zIGRpZmZlciIpCiAgICByZXR1cm4gewogICAgICAgICJmZWF0dXJlIjogImhhaXJzdHlsZSIsICJzdHlsZV9pZCI6IHN0eWxlX2lkLAogICAgICAgICJhZGFwdGVyX2lkIjogcFsiYWRhcHRlcl9pZCJdLCAiYWRhcHRlcl9zaGEyNTYiOiBwWyJhZGFwdGVyX3NoYTI1NiJdLAogICAgICAgICJhZGFwdGVyX3N0ZXBzIjogaW5mb1sidHJhaW5pbmdfc3RlcHMiXSwgImJhc2VfbW9kZWxfaWQiOiBSRUdJU1RSWVsiYmFzZV9tb2RlbF9pZCJdLAogICAgICAgICJiYXNlX21vZGVsX3JldmlzaW9uIjogcFsiYmFzZV9tb2RlbF9yZXZpc2lvbiJdLAogICAgICAgICJnZW5lcmF0b3IiOiAiZmx1eDJfa2xlaW5fYmFzZV8iICsgcFsiYWRhcHRlcl9pZCJdICsgIl9wcm9maWxlIiwgInByb21wdCI6IHBbInByb21wdCJdLAogICAgICAgICJleGVjdXRpb25fcHJvZmlsZSI6IGMucHJvZmlsZV9pZCwgImV4ZWN1dGlvbl9wcm9maWxlX3NoYTI1NiI6IGMucHJvZmlsZV9zaGEyNTYsCiAgICAgICAgInByb21wdF9zaGEyNTYiOiBzaGEyNTYocFsicHJvbXB0Il0uZW5jb2RlKCkpLmhleGRpZ2VzdCgpLAogICAgICAgICJzdHlsZV9yZWxlYXNlX2lkIjogUkVMRUFTRS5yZWxlYXNlX2lkLCAic3R5bGVfbWFuaWZlc3Rfc2hhMjU2IjogUkVMRUFTRS5tYW5pZmVzdF9zaGEyNTYsCiAgICAgICAgInRyYWluZWRfc3R5bGUiOiBGYWxzZSwgInNlZWQiOiAxOTc3LCAic3RlcHMiOiAyMCwgImd1aWRhbmNlIjogNC4wLAogICAgfQoKCmRlZiB2YWxpZGF0ZV9yZW1vdGVfcmVzdWx0KHBheWxvYWQsIHN0eWxlX2lkKToKICAgIGV4cGVjdGVkID0gZXhwZWN0ZWRfY29udHJhY3Qoc3R5bGVfaWQpCiAgICBtZXRhZGF0YSA9IHBheWxvYWQuZ2V0KCJtZXRhZGF0YSIpCiAgICBpbWFnZSA9IHBheWxvYWQuZ2V0KCJpbWFnZSIsIHt9KQogICAgaWYgKG5vdCBpc2luc3RhbmNlKGltYWdlLCBkaWN0KSBvciBpbWFnZS5nZXQoIndpZHRoIikgIT0gNTEyIG9yIGltYWdlLmdldCgiaGVpZ2h0IikgIT0gNTEyCiAgICAgICAgICAgIG9yIGltYWdlLmdldCgiY29udGVudF90eXBlIikgIT0gImltYWdlL3BuZyIpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkFkZGl0aW9uYWwgSGFpciByZXNwb25zZSBoYXMgd3JvbmcgaW1hZ2UgZGltZW5zaW9ucy90eXBlIikKICAgIGlmIChub3QgaXNpbnN0YW5jZShtZXRhZGF0YSwgZGljdCkgb3IgcGF5bG9hZC5nZXQoInN0YXR1cyIpICE9ICJjb21wbGV0ZWQiCiAgICAgICAgICAgIG9yIHBheWxvYWQuZ2V0KCJnZW5lcmF0b3IiKSAhPSBleHBlY3RlZFsiZ2VuZXJhdG9yIl0KICAgICAgICAgICAgb3IgbWV0YWRhdGEuZ2V0KCJ0cmFpbmVkX3N0eWxlIikgaXMgbm90IEZhbHNlCiAgICAgICAgICAgIG9yIGFueShtZXRhZGF0YS5nZXQoaykgIT0gdiBmb3IgaywgdiBpbiBleHBlY3RlZC5pdGVtcygpIGlmIGsgbm90IGluIHsiZ2VuZXJhdG9yIiwgInByb21wdCJ9KQogICAgICAgICAgICBvciAicHJvbXB0IiBpbiBtZXRhZGF0YSBhbmQgbWV0YWRhdGFbInByb21wdCJdICE9IGV4cGVjdGVkWyJwcm9tcHQiXSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQWRkaXRpb25hbCBIYWlyIHJlc3BvbnNlIGhhcyB3cm9uZyBwcm9maWxlL2FydGlmYWN0IHByb3ZlbmFuY2UiKQoKCmRlZiB3b3JrZXJfY2FwYWJpbGl0aWVzKG1vZGVsX3N0eWxlcywgcmVsZWFzZT1Ob25lKToKICAgIGZyb20gYXBwLnN0eWxlX3JlbGVhc2UgaW1wb3J0IFJFTEVBU0UKICAgIHJlbGVhc2UgPSBSRUxFQVNFIGlmIHJlbGVhc2UgaXMgTm9uZSBlbHNlIHJlbGVhc2UKICAgIGFkZGl0aW9ucyA9IGhhaXJfY2FuZGlkYXRlcyhyZWxlYXNlKQogICAgcmV0dXJuIHsic2NoZW1hIjogV09SS0VSX1NDSEVNQSwgInJlbGVhc2VfaWQiOiByZWxlYXNlLnJlbGVhc2VfaWQsCiAgICAgICAgICAgICJtYW5pZmVzdF9zaGEyNTYiOiByZWxlYXNlLm1hbmlmZXN0X3NoYTI1NiwgIm1vZGVsX3Byb2ZpbGUiOiBtb2RlbF9wcm9maWxlKHJlbGVhc2UpLAogICAgICAgICAgICAibW9kZWxfc3R5bGVzIjogbW9kZWxfc3R5bGVzLAogICAgICAgICAgICAicHJvZmlsZV9zaGEyNTYiOiB7Yy5wcm9maWxlX2lkOiBjLnByb2ZpbGVfc2hhMjU2IGZvciBjIGluIGFkZGl0aW9uc319Cg==', 'sha256': '2503e4b215fdce8a17054118314ad5d30c1afb143802d48a8d749d71de2d630f'}, 'backend/app/style_release.json': {'data': 'ewogICJzY2hlbWEiOiAiYmVhdXR5Y29yZS1zdHlsZS1yZWxlYXNlLXYxIiwKICAicmVsZWFzZV9pZCI6ICJzdHlsZXMtMjAyNjEwMzEtMDIiLAogICJjYW5kaWRhdGVzIjogWwogICAgewogICAgICAiZmVhdHVyZSI6ICJoYWlyc3R5bGUiLAogICAgICAiaWQiOiAiZXhwX2J1dHRlcmZseV9jdXQiLAogICAgICAibmFtZSI6ICJCdXR0ZXJmbHkgQ3V0IiwKICAgICAgImRlc2NyaXB0aW9uIjogIkJ1dHRlcmZseS1pbnNwaXJlZCBsYXllcnM7IHJldmlld2VkIHByZXZpZXdzIGNhbiBhcHBlYXIgc2hvcnRlciBhbmQgYm9iLWxpa2UuIFByZXZpZXcgbWF5IGNoYW5nZSBoYWlyIGNvbG9yLiIsCiAgICAgICJyb3V0ZSI6ICJoYWlyX3Byb21wdCIsCiAgICAgICJwcm9maWxlX2lkIjogImhhaXItYnV0dGVyZmx5LWN1dC12MSIsCiAgICAgICJwcm9maWxlX3NoYTI1NiI6ICI2YTYzZjMwOTQ2MGRmYjFiYzVjYmYwZmM3ZTM1ODRmYTY2ZmQyMGY1NjBhYmJjZWRmNWRjYjI1Y2Q4MzM4NzcwIiwKICAgICAgInJldmlld19zdGF0dXMiOiAiYWNjZXB0ZWQiLAogICAgICAiZW5hYmxlZCI6IHRydWUsCiAgICAgICJ0YWdzIjogWwogICAgICAgICJsYXllcmVkIiwKICAgICAgICAiZmFjZV9mcmFtaW5nIiwKICAgICAgICAidm9sdW1pbm91cyIsCiAgICAgICAgImV2ZXJ5ZGF5IgogICAgICBdLAogICAgICAic2VydmljZXMiOiBbCiAgICAgICAgIk1lbidzIEN1dCIsCiAgICAgICAgIldvbWVuJ3MgQ3V0ICYgQmxvd2RyeSIKICAgICAgXSwKICAgICAgImlucHV0X2NvbnN0cmFpbnRzIjogWwogICAgICAgICJBSSBwcmV2aWV3IG1heSBjaGFuZ2UgaGFpciBjb2xvciwgdGV4dHVyZSBvciBmcmluZ2UgcGxhY2VtZW50OyBjb25maXJtIHRoZSBmaW5hbCBjdXQgd2l0aCB5b3VyIHN0eWxpc3QuIgogICAgICBdLAogICAgICAiZXZpZGVuY2UiOiBbCiAgICAgICAgImRvY3MvZXhwZXJpbWVudHMvc3R5bGUtcmVsZWFzZS12YWxpZGF0aW9uLTIwMjYxMDI1LXJldmlldy5tZCIsCiAgICAgICAgImRvY3MvZXhwZXJpbWVudHMvaGFpci1leHBhbnNpb24tcmVzdWx0cy0yMDI2MTAzMS1yZXZpZXcubWQiLAogICAgICAgICJkb2NzL2V4cGVyaW1lbnRzL2FsbC1oYWlyLW5haWxzLWFwcHJvdmFsLTIwMjYxMDMxLmpzb24iCiAgICAgIF0KICAgIH0sCiAgICB7CiAgICAgICJmZWF0dXJlIjogImhhaXJzdHlsZSIsCiAgICAgICJpZCI6ICJleHBfd29sZl9jdXQiLAogICAgICAibmFtZSI6ICJXb2xmIEN1dCIsCiAgICAgICJkZXNjcmlwdGlvbiI6ICJBIGJvbGQsIHNwaWt5IGludGVycHJldGF0aW9uIG9mIGEgY2hvcHB5IHdvbGYgY3V0IHdpdGggYSBsb25nZXIgYmFjay4gUHJldmlldyBtYXkgY2hhbmdlIGhhaXIgY29sb3IuIiwKICAgICAgInJvdXRlIjogImhhaXJfcHJvbXB0IiwKICAgICAgInByb2ZpbGVfaWQiOiAiaGFpci13b2xmLWN1dC12MSIsCiAgICAgICJwcm9maWxlX3NoYTI1NiI6ICJhNzlmMzljYzdkM2U5MTdhMzkwZDlkZmM2MmRlNjVkNjkxODVkZThhNjhjNGRkYTA5ZTIxMDQ4YzlkMzJkMTIzIiwKICAgICAgInJldmlld19zdGF0dXMiOiAiYWNjZXB0ZWQiLAogICAgICAiZW5hYmxlZCI6IHRydWUsCiAgICAgICJ0YWdzIjogWwogICAgICAgICJjaG9wcHkiLAogICAgICAgICJ0ZXh0dXJlZCIsCiAgICAgICAgInNwaWt5IiwKICAgICAgICAid29sZiIsCiAgICAgICAgImJvbGQiCiAgICAgIF0sCiAgICAgICJzZXJ2aWNlcyI6IFsKICAgICAgICAiTWVuJ3MgQ3V0IiwKICAgICAgICAiV29tZW4ncyBDdXQgJiBCbG93ZHJ5IgogICAgICBdLAogICAgICAiaW5wdXRfY29uc3RyYWludHMiOiBbCiAgICAgICAgIkFJIHByZXZpZXcgbWF5IGNoYW5nZSBoYWlyIGNvbG9yLCB0ZXh0dXJlIG9yIGZyaW5nZSBwbGFjZW1lbnQ7IGNvbmZpcm0gdGhlIGZpbmFsIGN1dCB3aXRoIHlvdXIgc3R5bGlzdC4iCiAgICAgIF0sCiAgICAgICJldmlkZW5jZSI6IFsKICAgICAgICAiZG9jcy9leHBlcmltZW50cy9oYWlyLWV4cGFuc2lvbi1yZXN1bHRzLTIwMjYxMDMxLXJldmlldy5tZCIsCiAgICAgICAgImRvY3MvZXhwZXJpbWVudHMvYWxsLWhhaXItbmFpbHMtYXBwcm92YWwtMjAyNjEwMzEuanNvbiIKICAgICAgXQogICAgfSwKICAgIHsKICAgICAgImZlYXR1cmUiOiAiaGFpcnN0eWxlIiwKICAgICAgImlkIjogImV4cF9rb3JlYW5fdHdvX2Jsb2NrIiwKICAgICAgIm5hbWUiOiAiS29yZWFuIFR3by1CbG9jayIsCiAgICAgICJkZXNjcmlwdGlvbiI6ICJBIGZ1bGwgdXBwZXIgc2VjdGlvbiBhbmQgZGVuc2UgZnJpbmdlIGluc3BpcmVkIGJ5IHRoZSBLb3JlYW4gdHdvLWJsb2NrIGN1dC4gUHJldmlldyBtYXkgY292ZXIgZXllcyBvciBjaGFuZ2UgaGFpciBjb2xvci4iLAogICAgICAicm91dGUiOiAiaGFpcl9wcm9tcHQiLAogICAgICAicHJvZmlsZV9pZCI6ICJoYWlyLWtvcmVhbi10d28tYmxvY2stdjEiLAogICAgICAicHJvZmlsZV9zaGEyNTYiOiAiMDkxM2Q1NGRiNDA4ZmE3OWU5MDg0YTIwOGM0MDQ5YWZiNThkOTY1MTA5MTEwNzM5ODM3MWU1M2Y4MmJjMTBjMSIsCiAgICAgICJyZXZpZXdfc3RhdHVzIjogImFjY2VwdGVkIiwKICAgICAgImVuYWJsZWQiOiB0cnVlLAogICAgICAidGFncyI6IFsKICAgICAgICAiZnJpbmdlIiwKICAgICAgICAidHdvX2Jsb2NrIiwKICAgICAgICAia29yZWFuIiwKICAgICAgICAiZXZlcnlkYXkiCiAgICAgIF0sCiAgICAgICJzZXJ2aWNlcyI6IFsKICAgICAgICAiTWVuJ3MgQ3V0IiwKICAgICAgICAiV29tZW4ncyBDdXQgJiBCbG93ZHJ5IgogICAgICBdLAogICAgICAiaW5wdXRfY29uc3RyYWludHMiOiBbCiAgICAgICAgIkFJIHByZXZpZXcgbWF5IGNoYW5nZSBoYWlyIGNvbG9yLCB0ZXh0dXJlIG9yIGZyaW5nZSBwbGFjZW1lbnQ7IGNvbmZpcm0gdGhlIGZpbmFsIGN1dCB3aXRoIHlvdXIgc3R5bGlzdC4iCiAgICAgIF0sCiAgICAgICJldmlkZW5jZSI6IFsKICAgICAgICAiZG9jcy9leHBlcmltZW50cy9zdHlsZS1yZWxlYXNlLXZhbGlkYXRpb24tMjAyNjEwMjUtcmV2aWV3Lm1kIiwKICAgICAgICAiZG9jcy9leHBlcmltZW50cy9oYWlyLWV4cGFuc2lvbi1yZXN1bHRzLTIwMjYxMDMxLXJldmlldy5tZCIsCiAgICAgICAgImRvY3MvZXhwZXJpbWVudHMvYWxsLWhhaXItbmFpbHMtYXBwcm92YWwtMjAyNjEwMzEuanNvbiIKICAgICAgXQogICAgfSwKICAgIHsKICAgICAgImZlYXR1cmUiOiAiaGFpcnN0eWxlIiwKICAgICAgImlkIjogImV4cF9jb21tYV9oYWlyIiwKICAgICAgIm5hbWUiOiAiS29yZWFuIENvbW1hIEhhaXIiLAogICAgICAiZGVzY3JpcHRpb24iOiAiQSBzaWRlLXBhcnRlZCwgaW53YXJkLWN1cnZlZCBmcmluZ2UgaW5zcGlyZWQgYnkgS29yZWFuIGNvbW1hIGhhaXIuIFByZXZpZXcgbWF5IHJlc2VtYmxlIGN1cnRhaW4gaGFpciwgY292ZXIgZXllcyBvciBjaGFuZ2UgaGFpciBjb2xvci4iLAogICAgICAicm91dGUiOiAiaGFpcl9wcm9tcHQiLAogICAgICAicHJvZmlsZV9pZCI6ICJoYWlyLWNvbW1hLWhhaXItdjEiLAogICAgICAicHJvZmlsZV9zaGEyNTYiOiAiZDdjYzVhYjg4NzBjZmM5ZDAxNGM2NWMyYjVkOWM1OGY1Y2U1YTVlODdmNzg1MTE2MzYwNTdjMzYyMDg4NzY2OCIsCiAgICAgICJyZXZpZXdfc3RhdHVzIjogImFjY2VwdGVkIiwKICAgICAgImVuYWJsZWQiOiB0cnVlLAogICAgICAidGFncyI6IFsKICAgICAgICAic2lkZV9wYXJ0IiwKICAgICAgICAiY3VydmVkIiwKICAgICAgICAiZnJpbmdlIiwKICAgICAgICAia29yZWFuIiwKICAgICAgICAiZXZlcnlkYXkiCiAgICAgIF0sCiAgICAgICJzZXJ2aWNlcyI6IFsKICAgICAgICAiTWVuJ3MgQ3V0IiwKICAgICAgICAiV29tZW4ncyBDdXQgJiBCbG93ZHJ5IgogICAgICBdLAogICAgICAiaW5wdXRfY29uc3RyYWludHMiOiBbCiAgICAgICAgIkFJIHByZXZpZXcgbWF5IGNoYW5nZSBoYWlyIGNvbG9yLCB0ZXh0dXJlIG9yIGZyaW5nZSBwbGFjZW1lbnQ7IGNvbmZpcm0gdGhlIGZpbmFsIGN1dCB3aXRoIHlvdXIgc3R5bGlzdC4iCiAgICAgIF0sCiAgICAgICJldmlkZW5jZSI6IFsKICAgICAgICAiZG9jcy9leHBlcmltZW50cy9zdHlsZS1yZWxlYXNlLXZhbGlkYXRpb24tMjAyNjEwMjUtcmV2aWV3Lm1kIiwKICAgICAgICAiZG9jcy9leHBlcmltZW50cy9oYWlyLWV4cGFuc2lvbi1yZXN1bHRzLTIwMjYxMDMxLXJldmlldy5tZCIsCiAgICAgICAgImRvY3MvZXhwZXJpbWVudHMvYWxsLWhhaXItbmFpbHMtYXBwcm92YWwtMjAyNjEwMzEuanNvbiIKICAgICAgXQogICAgfSwKICAgIHsKICAgICAgImZlYXR1cmUiOiAiaGFpcnN0eWxlIiwKICAgICAgImlkIjogImV4cF9mcmVuY2hfYm9iX3dpc3B5IiwKICAgICAgIm5hbWUiOiAiVGV4dHVyZWQgRnJlbmNoIEJvYiB3aXRoIFdpc3B5IEJhbmdzIiwKICAgICAgImRlc2NyaXB0aW9uIjogIkEgY3VybHksIGphdy1sZW5ndGggdGV4dHVyZWQgYm9iIHdpdGggc2VwYXJhdGVkIHdpc3B5IGJhbmdzLiBQcmV2aWV3IG1heSBjaGFuZ2UgaGFpciBjb2xvci4iLAogICAgICAicm91dGUiOiAiaGFpcl9wcm9tcHQiLAogICAgICAicHJvZmlsZV9pZCI6ICJoYWlyLWZyZW5jaC1ib2ItdjEiLAogICAgICAicHJvZmlsZV9zaGEyNTYiOiAiZmVjNDNjMzFhMDRlMWNlYTdiMTg3NjYwMjZmMDg4ZDYzNTM0NDI5ZGRmYzZhZWQ5ZDI3MjNiNGE4NmExOGQwZCIsCiAgICAgICJyZXZpZXdfc3RhdHVzIjogImFjY2VwdGVkIiwKICAgICAgImVuYWJsZWQiOiB0cnVlLAogICAgICAidGFncyI6IFsKICAgICAgICAic2hvcnQiLAogICAgICAgICJib2IiLAogICAgICAgICJ0ZXh0dXJlZCIsCiAgICAgICAgImN1cmx5IiwKICAgICAgICAid2lzcHkiLAogICAgICAgICJiYW5ncyIsCiAgICAgICAgImV2ZXJ5ZGF5IgogICAgICBdLAogICAgICAic2VydmljZXMiOiBbCiAgICAgICAgIk1lbidzIEN1dCIsCiAgICAgICAgIldvbWVuJ3MgQ3V0ICYgQmxvd2RyeSIKICAgICAgXSwKICAgICAgImlucHV0X2NvbnN0cmFpbnRzIjogWwogICAgICAgICJBSSBwcmV2aWV3IG1heSBjaGFuZ2UgaGFpciBjb2xvciBhbmQgdGV4dHVyZTsgZmluYWwgc2Fsb24gcmVzdWx0cyByZXF1aXJlIHN0eWxpc3QgY29uc3VsdGF0aW9uLiIKICAgICAgXSwKICAgICAgImV2aWRlbmNlIjogWwogICAgICAgICJkb2NzL2V4cGVyaW1lbnRzL2hhaXItZXhwYW5zaW9uLXJlc3VsdHMtMjAyNjEwMzEtcmV2aWV3Lm1kIiwKICAgICAgICAiZG9jcy9leHBlcmltZW50cy9mcmVuY2gtYm9iLWFwcHJvdmFsLTIwMjYxMDMxLmpzb24iCiAgICAgIF0KICAgIH0sCiAgICB7CiAgICAgICJmZWF0dXJlIjogIm1ha2V1cCIsCiAgICAgICJpZCI6ICJleHBfY2xvdWRfYmVycnlfbGlwcyIsCiAgICAgICJuYW1lIjogIkNsb3VkLVNraW4gTWFrZXVwIHdpdGggQmx1cnJlZCBCZXJyeSBMaXBzIiwKICAgICAgImRlc2NyaXB0aW9uIjogIkEgc29mdCBiYXNlIHdpdGggZGlmZnVzZWQgYmVycnkgbGlwIGVkZ2VzIGFuZCByZXN0cmFpbmVkIGNoZWVrIGNvbG9yLiIsCiAgICAgICJyb3V0ZSI6ICJtYWtldXBfcHJvZmlsZSIsCiAgICAgICJwcm9maWxlX2lkIjogbnVsbCwKICAgICAgInByb2ZpbGVfc2hhMjU2IjogbnVsbCwKICAgICAgInJldmlld19zdGF0dXMiOiAiaGVsZCIsCiAgICAgICJlbmFibGVkIjogZmFsc2UsCiAgICAgICJ0YWdzIjogWwogICAgICAgICJleHBlcmltZW50YWwiCiAgICAgIF0sCiAgICAgICJzZXJ2aWNlcyI6IFsKICAgICAgICAiTWFrZXVwIEFwcGxpY2F0aW9uIgogICAgICBdLAogICAgICAiaW5wdXRfY29uc3RyYWludHMiOiBbXSwKICAgICAgImV2aWRlbmNlIjogWwogICAgICAgICJkb2NzL2V4cGVyaW1lbnRzL3N0eWxlLXJlbGVhc2UtdmFsaWRhdGlvbi0yMDI2MTAyNS1yZXZpZXcubWQiCiAgICAgIF0KICAgIH0sCiAgICB7CiAgICAgICJmZWF0dXJlIjogIm1ha2V1cCIsCiAgICAgICJpZCI6ICJleHBfbWFuZ2FfbGFzaCIsCiAgICAgICJuYW1lIjogIk1hbmdhLUxhc2ggTWFrZXVwIiwKICAgICAgImRlc2NyaXB0aW9uIjogIlNlcGFyYXRlZCBsYXNoIGNsdXN0ZXJzIHdpdGggZGVsaWNhdGUgbGluZXIsIHNvZnQgYmx1c2gsIGFuZCBhIHJlc3RyYWluZWQgbGlwLiIsCiAgICAgICJyb3V0ZSI6ICJtYWtldXBfcHJvZmlsZSIsCiAgICAgICJwcm9maWxlX2lkIjogbnVsbCwKICAgICAgInByb2ZpbGVfc2hhMjU2IjogbnVsbCwKICAgICAgInJldmlld19zdGF0dXMiOiAiaGVsZCIsCiAgICAgICJlbmFibGVkIjogZmFsc2UsCiAgICAgICJ0YWdzIjogWwogICAgICAgICJleHBlcmltZW50YWwiCiAgICAgIF0sCiAgICAgICJzZXJ2aWNlcyI6IFsKICAgICAgICAiTWFrZXVwIEFwcGxpY2F0aW9uIgogICAgICBdLAogICAgICAiaW5wdXRfY29uc3RyYWludHMiOiBbXSwKICAgICAgImV2aWRlbmNlIjogWwogICAgICAgICJkb2NzL2V4cGVyaW1lbnRzL21ha2V1cC1leHBhbnNpb24tcmVzdWx0cy0yMDI2MTAzMS1yZXZpZXcubWQiCiAgICAgIF0KICAgIH0sCiAgICB7CiAgICAgICJmZWF0dXJlIjogIm1ha2V1cCIsCiAgICAgICJpZCI6ICJleHBfaWN5X3NpbHZlciIsCiAgICAgICJuYW1lIjogIkljeSBTaWx2ZXIgRXllIE1ha2V1cCIsCiAgICAgICJkZXNjcmlwdGlvbiI6ICJMb2NhbGl6ZWQgc2lsdmVyLWJsdWUgZXllbGlkIHNoaW1tZXIgYmFsYW5jZWQgd2l0aCBzdWJkdWVkIGNoZWVrcyBhbmQgbGlwcy4iLAogICAgICAicm91dGUiOiAibWFrZXVwX3Byb2ZpbGUiLAogICAgICAicHJvZmlsZV9pZCI6IG51bGwsCiAgICAgICJwcm9maWxlX3NoYTI1NiI6IG51bGwsCiAgICAgICJyZXZpZXdfc3RhdHVzIjogImhlbGQiLAogICAgICAiZW5hYmxlZCI6IGZhbHNlLAogICAgICAidGFncyI6IFsKICAgICAgICAiZXhwZXJpbWVudGFsIgogICAgICBdLAogICAgICAic2VydmljZXMiOiBbCiAgICAgICAgIk1ha2V1cCBBcHBsaWNhdGlvbiIKICAgICAgXSwKICAgICAgImlucHV0X2NvbnN0cmFpbnRzIjogW10sCiAgICAgICJldmlkZW5jZSI6IFsKICAgICAgICAiZG9jcy9leHBlcmltZW50cy9zdHlsZS1yZWxlYXNlLXZhbGlkYXRpb24tMjAyNjEwMjUtcmV2aWV3Lm1kIgogICAgICBdCiAgICB9LAogICAgewogICAgICAiZmVhdHVyZSI6ICJuYWlscyIsCiAgICAgICJpZCI6ICJleHBfcmFzcGJlcnJ5X2plbGx5IiwKICAgICAgIm5hbWUiOiAiUmFzcGJlcnJ5IEplbGx5IiwKICAgICAgImRlc2NyaXB0aW9uIjogIkEgdHJhbnNsdWNlbnQgcmFzcGJlcnJ5IHRpbnQgdGhhdCByZXRhaW5zIHRoZSBwaG90b2dyYXBoZWQgbmFpbCBkZXRhaWwuIiwKICAgICAgInJvdXRlIjogIm5haWxzX3JlbmRlcmVyIiwKICAgICAgInByb2ZpbGVfaWQiOiAibmFpbHMtZXhwYW5kZWQtdjEiLAogICAgICAicHJvZmlsZV9zaGEyNTYiOiAiYWYwNTJiYWY5M2RlNjc0OTNiMzk2MjQ0NjM5NDNiMGE0YjA1MmUzOWM0ZGYyZTA4MDRmNGNkYjViYzgyZTdjNCIsCiAgICAgICJyZXZpZXdfc3RhdHVzIjogImFjY2VwdGVkIiwKICAgICAgImVuYWJsZWQiOiB0cnVlLAogICAgICAidGFncyI6IFsKICAgICAgICAicmFzcGJlcnJ5IiwKICAgICAgICAiamVsbHkiLAogICAgICAgICJ0cmFuc2x1Y2VudCIKICAgICAgXSwKICAgICAgInNlcnZpY2VzIjogWwogICAgICAgICJDbGFzc2ljIE1hbmljdXJlIiwKICAgICAgICAiR2VsIE1hbmljdXJlIiwKICAgICAgICAiTmFpbCBBcnQgKHBlciBuYWlsKSIKICAgICAgXSwKICAgICAgImlucHV0X2NvbnN0cmFpbnRzIjogW10sCiAgICAgICJldmlkZW5jZSI6IFsKICAgICAgICAiZG9jcy9leHBlcmltZW50cy9zdHlsZS1yZWxlYXNlLXZhbGlkYXRpb24tMjAyNjEwMjUtcmV2aWV3Lm1kIiwKICAgICAgICAiZG9jcy9leHBlcmltZW50cy9leHBhbmRlZC1uYWlscy1hcHByb3ZhbC0yMDI2MTAzMC5qc29uIgogICAgICBdCiAgICB9LAogICAgewogICAgICAiZmVhdHVyZSI6ICJuYWlscyIsCiAgICAgICJpZCI6ICJleHBfcGlua19hdXJhIiwKICAgICAgIm5hbWUiOiAiUGluayBBdXJhIE9tYnLDqSIsCiAgICAgICJkZXNjcmlwdGlvbiI6ICJBIHNvZnQgcGluayBiYXNlIHdpdGggZGVlcGVyIHBpbmsgY29uY2VudHJhdGVkIG5lYXIgdGhlIG5haWwgY2VudHJlLiIsCiAgICAgICJyb3V0ZSI6ICJuYWlsc19yZW5kZXJlciIsCiAgICAgICJwcm9maWxlX2lkIjogIm5haWxzLWV4cGFuZGVkLXYxIiwKICAgICAgInByb2ZpbGVfc2hhMjU2IjogImFmMDUyYmFmOTNkZTY3NDkzYjM5NjI0NDYzOTQzYjBhNGIwNTJlMzljNGRmMmUwODA0ZjRjZGI1YmM4MmU3YzQiLAogICAgICAicmV2aWV3X3N0YXR1cyI6ICJhY2NlcHRlZCIsCiAgICAgICJlbmFibGVkIjogdHJ1ZSwKICAgICAgInRhZ3MiOiBbCiAgICAgICAgInBpbmsiLAogICAgICAgICJhdXJhIiwKICAgICAgICAib21icmUiCiAgICAgIF0sCiAgICAgICJzZXJ2aWNlcyI6IFsKICAgICAgICAiQ2xhc3NpYyBNYW5pY3VyZSIsCiAgICAgICAgIkdlbCBNYW5pY3VyZSIsCiAgICAgICAgIk5haWwgQXJ0IChwZXIgbmFpbCkiCiAgICAgIF0sCiAgICAgICJpbnB1dF9jb25zdHJhaW50cyI6IFtdLAogICAgICAiZXZpZGVuY2UiOiBbCiAgICAgICAgImRvY3MvZXhwZXJpbWVudHMvc3R5bGUtcmVsZWFzZS12YWxpZGF0aW9uLTIwMjYxMDI1LXJldmlldy5tZCIsCiAgICAgICAgImRvY3MvZXhwZXJpbWVudHMvZXhwYW5kZWQtbmFpbHMtYXBwcm92YWwtMjAyNjEwMzAuanNvbiIKICAgICAgXQogICAgfSwKICAgIHsKICAgICAgImZlYXR1cmUiOiAibmFpbHMiLAogICAgICAiaWQiOiAiZXhwX25lZ2F0aXZlX3NwYWNlX21pY3JvZG90cyIsCiAgICAgICJuYW1lIjogIk5lZ2F0aXZlLVNwYWNlIE1pY3JvLURvdHMiLAogICAgICAiZGVzY3JpcHRpb24iOiAiU3BhcnNlIGRhcmsgbWljcm8tZG90cyBvdmVyIGEgbGlnaHRseSB0aW50ZWQgbmF0dXJhbCBuYWlsLiIsCiAgICAgICJyb3V0ZSI6ICJuYWlsc19yZW5kZXJlciIsCiAgICAgICJwcm9maWxlX2lkIjogIm5haWxzLWV4cGFuZGVkLXYxIiwKICAgICAgInByb2ZpbGVfc2hhMjU2IjogImFmMDUyYmFmOTNkZTY3NDkzYjM5NjI0NDYzOTQzYjBhNGIwNTJlMzljNGRmMmUwODA0ZjRjZGI1YmM4MmU3YzQiLAogICAgICAicmV2aWV3X3N0YXR1cyI6ICJhY2NlcHRlZCIsCiAgICAgICJlbmFibGVkIjogdHJ1ZSwKICAgICAgInRhZ3MiOiBbCiAgICAgICAgImRvdHMiLAogICAgICAgICJtaW5pbWFsIiwKICAgICAgICAibmVnYXRpdmVfc3BhY2UiCiAgICAgIF0sCiAgICAgICJzZXJ2aWNlcyI6IFsKICAgICAgICAiQ2xhc3NpYyBNYW5pY3VyZSIsCiAgICAgICAgIkdlbCBNYW5pY3VyZSIsCiAgICAgICAgIk5haWwgQXJ0IChwZXIgbmFpbCkiCiAgICAgIF0sCiAgICAgICJpbnB1dF9jb25zdHJhaW50cyI6IFtdLAogICAgICAiZXZpZGVuY2UiOiBbCiAgICAgICAgImRvY3MvZXhwZXJpbWVudHMvc3R5bGUtcmVsZWFzZS12YWxpZGF0aW9uLTIwMjYxMDI1LXJldmlldy5tZCIsCiAgICAgICAgImRvY3MvZXhwZXJpbWVudHMvZXhwYW5kZWQtbmFpbHMtYXBwcm92YWwtMjAyNjEwMzAuanNvbiIKICAgICAgXQogICAgfSwKICAgIHsKICAgICAgImZlYXR1cmUiOiAibmFpbHMiLAogICAgICAiaWQiOiAiZXhwX21pbGt5X3BlYXJsX2Nocm9tZSIsCiAgICAgICJuYW1lIjogIk1pbGt5IFBlYXJsIENocm9tZSIsCiAgICAgICJkZXNjcmlwdGlvbiI6ICJBIHBhbGUgYmFzZSB3aXRoIGEgc3VidGxlIHBlYXJsZXNjZW50IHJlZmxlY3RpdmUgb3ZlcmxheS4iLAogICAgICAicm91dGUiOiAibmFpbHNfcmVuZGVyZXIiLAogICAgICAicHJvZmlsZV9pZCI6IG51bGwsCiAgICAgICJwcm9maWxlX3NoYTI1NiI6IG51bGwsCiAgICAgICJyZXZpZXdfc3RhdHVzIjogInVudGVzdGVkIiwKICAgICAgImVuYWJsZWQiOiBmYWxzZSwKICAgICAgInRhZ3MiOiBbCiAgICAgICAgInBlYXJsIiwKICAgICAgICAiY2hyb21lIiwKICAgICAgICAibWlsa3kiCiAgICAgIF0sCiAgICAgICJzZXJ2aWNlcyI6IFsKICAgICAgICAiQ2xhc3NpYyBNYW5pY3VyZSIsCiAgICAgICAgIkdlbCBNYW5pY3VyZSIsCiAgICAgICAgIk5haWwgQXJ0IChwZXIgbmFpbCkiCiAgICAgIF0sCiAgICAgICJpbnB1dF9jb25zdHJhaW50cyI6IFtdLAogICAgICAiZXZpZGVuY2UiOiBbXQogICAgfSwKICAgIHsKICAgICAgImZlYXR1cmUiOiAibmFpbHMiLAogICAgICAiaWQiOiAiZXhwX2RhcmtfcmVkX3ZlbHZldF9jYXRfZXllIiwKICAgICAgIm5hbWUiOiAiRGFyay1SZWQgVmVsdmV0IENhdC1FeWUiLAogICAgICAiZGVzY3JpcHRpb24iOiAiQSBkZWVwIHJlZCBiYXNlIHdpdGggYSBuYXJyb3cgcmVmbGVjdGl2ZSBiYW5kIGFuZCBhIHNvZnQgbHVtaW5vdXMgZmluaXNoLiIsCiAgICAgICJyb3V0ZSI6ICJuYWlsc19yZW5kZXJlciIsCiAgICAgICJwcm9maWxlX2lkIjogbnVsbCwKICAgICAgInByb2ZpbGVfc2hhMjU2IjogbnVsbCwKICAgICAgInJldmlld19zdGF0dXMiOiAidW50ZXN0ZWQiLAogICAgICAiZW5hYmxlZCI6IGZhbHNlLAogICAgICAidGFncyI6IFsKICAgICAgICAicmVkIiwKICAgICAgICAidmVsdmV0IiwKICAgICAgICAiY2F0X2V5ZSIKICAgICAgXSwKICAgICAgInNlcnZpY2VzIjogWwogICAgICAgICJDbGFzc2ljIE1hbmljdXJlIiwKICAgICAgICAiR2VsIE1hbmljdXJlIiwKICAgICAgICAiTmFpbCBBcnQgKHBlciBuYWlsKSIKICAgICAgXSwKICAgICAgImlucHV0X2NvbnN0cmFpbnRzIjogW10sCiAgICAgICJldmlkZW5jZSI6IFtdCiAgICB9CiAgXQp9Cg==', 'sha256': '61cce06d4e76a6570550b2fee4c7ff4183323aea0ce1b3a1b63e191fced118b5'}, 'backend/app/style_release.py': {'data': 'IiIiVmVyc2lvbmVkIGV4cGFuc2lvbiBhcHByb3ZhbCBkYXRhLiBGcm96ZW4gbW9kZWwgbWV0YWRhdGEgaXMgbmV2ZXIgcmV3cml0dGVuLiIiIg0KDQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MNCmZyb20gaGFzaGxpYiBpbXBvcnQgc2hhMjU2DQppbXBvcnQganNvbg0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQppbXBvcnQgcmUNCg0KUEFUSCA9IFBhdGgoX19maWxlX18pLndpdGhfc3VmZml4KCIuanNvbiIpDQpTQ0hFTUEgPSAiYmVhdXR5Y29yZS1zdHlsZS1yZWxlYXNlLXYxIg0KRkVBVFVSRVMgPSAoImhhaXJzdHlsZSIsICJtYWtldXAiLCAibmFpbHMiKQ0KUk9VVEVTID0geyJoYWlyc3R5bGUiOiAiaGFpcl9wcm9tcHQiLCAibWFrZXVwIjogIm1ha2V1cF9wcm9maWxlIiwgIm5haWxzIjogIm5haWxzX3JlbmRlcmVyIn0NClJFVklFV1MgPSBmcm96ZW5zZXQoKCJ1bnRlc3RlZCIsICJoZWxkIiwgInBlbmRpbmdfdmlzdWFsIiwgImFjY2VwdGVkIikpDQpJRCA9IHJlLmNvbXBpbGUociJbYS16XVthLXowLTlfLV17MCw3OX1cWiIpDQpIQVNIID0gcmUuY29tcGlsZShyIlswLTlhLWZdezY0fVxaIikNCg0KDQpkZWYgY2Fub25pY2FsX2RpZ2VzdCh2YWx1ZSkgLT4gc3RyOg0KICAgIHJldHVybiBzaGEyNTYoanNvbi5kdW1wcyh2YWx1ZSwgc29ydF9rZXlzPVRydWUsIGVuc3VyZV9hc2NpaT1UcnVlLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzZXBhcmF0b3JzPSgiLCIsICI6IikpLmVuY29kZSgpKS5oZXhkaWdlc3QoKQ0KDQoNCmRlZiBwcm9maWxlX2RpZ2VzdChwYXRoOiBQYXRoKSAtPiBzdHI6DQogICAgIiIiQ2Fub25pY2FsIExGIHNvdXJjZSBkaWdlc3Q7IGFyY2hpdmUgaW52ZW50b3JpZXMgc3RpbGwgaGFzaCBleGFjdCBieXRlcy4iIiINCiAgICByZXR1cm4gc2hhMjU2KHBhdGgucmVhZF9ieXRlcygpLnJlcGxhY2UoYiJcclxuIiwgYiJcbiIpKS5oZXhkaWdlc3QoKQ0KDQoNCmRlZiBfdW5pcXVlX29iamVjdChwYWlycyk6DQogICAgdmFsdWUgPSB7fQ0KICAgIGZvciBrZXksIGl0ZW0gaW4gcGFpcnM6DQogICAgICAgIGlmIGtleSBpbiB2YWx1ZToNCiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkR1cGxpY2F0ZSBzdHlsZSByZWxlYXNlIGZpZWxkIikNCiAgICAgICAgdmFsdWVba2V5XSA9IGl0ZW0NCiAgICByZXR1cm4gdmFsdWUNCg0KDQpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQ0KY2xhc3MgQ2FuZGlkYXRlOg0KICAgIGZlYXR1cmU6IHN0cg0KICAgIGlkOiBzdHINCiAgICBuYW1lOiBzdHINCiAgICBkZXNjcmlwdGlvbjogc3RyDQogICAgcm91dGU6IHN0cg0KICAgIHByb2ZpbGVfaWQ6IHN0ciB8IE5vbmUNCiAgICBwcm9maWxlX3NoYTI1Njogc3RyIHwgTm9uZQ0KICAgIHJldmlld19zdGF0dXM6IHN0cg0KICAgIGVuYWJsZWQ6IGJvb2wNCiAgICB0YWdzOiB0dXBsZVtzdHIsIC4uLl0NCiAgICBzZXJ2aWNlczogdHVwbGVbc3RyLCAuLi5dDQogICAgaW5wdXRfY29uc3RyYWludHM6IHR1cGxlW3N0ciwgLi4uXQ0KICAgIGV2aWRlbmNlOiB0dXBsZVtzdHIsIC4uLl0NCg0KDQpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQ0KY2xhc3MgU3R5bGVSZWxlYXNlOg0KICAgIHJlbGVhc2VfaWQ6IHN0cg0KICAgIG1hbmlmZXN0X3NoYTI1Njogc3RyDQogICAgY2FuZGlkYXRlczogdHVwbGVbQ2FuZGlkYXRlLCAuLi5dDQoNCiAgICBkZWYgZW5hYmxlZChzZWxmLCBmZWF0dXJlOiBzdHIgfCBOb25lID0gTm9uZSkgLT4gdHVwbGVbQ2FuZGlkYXRlLCAuLi5dOg0KICAgICAgICByZXR1cm4gdHVwbGUoYyBmb3IgYyBpbiBzZWxmLmNhbmRpZGF0ZXMgaWYgYy5lbmFibGVkDQogICAgICAgICAgICAgICAgICAgICBhbmQgKGZlYXR1cmUgaXMgTm9uZSBvciBjLmZlYXR1cmUgPT0gZmVhdHVyZSkpDQoNCg0KZGVmIF9zdHJpbmdzKHZhbHVlLCBmaWVsZCwgKiwgcmVxdWlyZWQ9RmFsc2UsIGxpbWl0PTI0MCk6DQogICAgaWYgKG5vdCBpc2luc3RhbmNlKHZhbHVlLCBsaXN0KSBvciBsZW4odmFsdWUpID4gMzANCiAgICAgICAgICAgIG9yIGFueShub3QgaXNpbnN0YW5jZShzLCBzdHIpIG9yIG5vdCBzLnN0cmlwKCkgb3IgbGVuKHMpID4gbGltaXQgZm9yIHMgaW4gdmFsdWUpDQogICAgICAgICAgICBvciBsZW4odmFsdWUpICE9IGxlbihzZXQodmFsdWUpKSBvciByZXF1aXJlZCBhbmQgbm90IHZhbHVlKToNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiSW52YWxpZCBzdHlsZSByZWxlYXNlICIgKyBmaWVsZCkNCiAgICByZXR1cm4gdHVwbGUodmFsdWUpDQoNCg0KZGVmIHBhcnNlX3JlbGVhc2UodmFsdWUpIC0+IFN0eWxlUmVsZWFzZToNCiAgICBpZiAobm90IGlzaW5zdGFuY2UodmFsdWUsIGRpY3QpIG9yIHNldCh2YWx1ZSkgIT0geyJzY2hlbWEiLCAicmVsZWFzZV9pZCIsICJjYW5kaWRhdGVzIn0NCiAgICAgICAgICAgIG9yIHZhbHVlWyJzY2hlbWEiXSAhPSBTQ0hFTUEgb3Igbm90IGlzaW5zdGFuY2UodmFsdWVbInJlbGVhc2VfaWQiXSwgc3RyKQ0KICAgICAgICAgICAgb3Igbm90IElELmZ1bGxtYXRjaCh2YWx1ZVsicmVsZWFzZV9pZCJdKSBvciBub3QgaXNpbnN0YW5jZSh2YWx1ZVsiY2FuZGlkYXRlcyJdLCBsaXN0KQ0KICAgICAgICAgICAgb3IgbGVuKHZhbHVlWyJjYW5kaWRhdGVzIl0pID4gMTAwKToNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiSW52YWxpZCBzdHlsZSByZWxlYXNlIGhlYWRlciIpDQogICAgZmllbGRzID0gc2V0KENhbmRpZGF0ZS5fX2RhdGFjbGFzc19maWVsZHNfXykNCiAgICBmb3VuZCA9IFtdDQogICAgc2VlbiA9IHNldCgpDQogICAgZm9yIHJvdyBpbiB2YWx1ZVsiY2FuZGlkYXRlcyJdOg0KICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShyb3csIGRpY3QpIG9yIHNldChyb3cpICE9IGZpZWxkczoNCiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkludmFsaWQgc3R5bGUgcmVsZWFzZSBjYW5kaWRhdGUgZmllbGRzIikNCiAgICAgICAgaWYgKG5vdCBpc2luc3RhbmNlKHJvd1siZmVhdHVyZSJdLCBzdHIpIG9yIHJvd1siZmVhdHVyZSJdIG5vdCBpbiBGRUFUVVJFUyBvciBub3QgaXNpbnN0YW5jZShyb3dbImlkIl0sIHN0cikNCiAgICAgICAgICAgICAgICBvciBub3QgSUQuZnVsbG1hdGNoKHJvd1siaWQiXSkgb3Igbm90IHJvd1siaWQiXS5zdGFydHN3aXRoKCJleHBfIikgb3Igcm93WyJpZCJdIGluIHNlZW4NCiAgICAgICAgICAgICAgICBvciByb3dbInJvdXRlIl0gIT0gUk9VVEVTW3Jvd1siZmVhdHVyZSJdXQ0KICAgICAgICAgICAgICAgIG9yIG5vdCBpc2luc3RhbmNlKHJvd1sicmV2aWV3X3N0YXR1cyJdLCBzdHIpIG9yIHJvd1sicmV2aWV3X3N0YXR1cyJdIG5vdCBpbiBSRVZJRVdTDQogICAgICAgICAgICAgICAgb3IgdHlwZShyb3dbImVuYWJsZWQiXSkgaXMgbm90IGJvb2wpOg0KICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiSW52YWxpZCBzdHlsZSByZWxlYXNlIGlkZW50aXR5LCByb3V0ZSBvciByZXZpZXciKQ0KICAgICAgICBmb3Iga2V5LCBsaW1pdCBpbiAoKCJuYW1lIiwgMjU1KSwgKCJkZXNjcmlwdGlvbiIsIDI0MCkpOg0KICAgICAgICAgICAgaWYgbm90IGlzaW5zdGFuY2Uocm93W2tleV0sIHN0cikgb3Igbm90IHJvd1trZXldLnN0cmlwKCkgb3IgbGVuKHJvd1trZXldKSA+IGxpbWl0Og0KICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkludmFsaWQgc3R5bGUgcmVsZWFzZSAiICsga2V5KQ0KICAgICAgICBwcm9maWxlLCBkaWdlc3QgPSByb3dbInByb2ZpbGVfaWQiXSwgcm93WyJwcm9maWxlX3NoYTI1NiJdDQogICAgICAgIGlmICgocHJvZmlsZSBpcyBOb25lKSAhPSAoZGlnZXN0IGlzIE5vbmUpDQogICAgICAgICAgICAgICAgb3IgcHJvZmlsZSBpcyBub3QgTm9uZSBhbmQgKG5vdCBpc2luc3RhbmNlKHByb2ZpbGUsIHN0cikgb3Igbm90IElELmZ1bGxtYXRjaChwcm9maWxlKSkNCiAgICAgICAgICAgICAgICBvciBkaWdlc3QgaXMgbm90IE5vbmUgYW5kIChub3QgaXNpbnN0YW5jZShkaWdlc3QsIHN0cikgb3Igbm90IEhBU0guZnVsbG1hdGNoKGRpZ2VzdCkpKToNCiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkludmFsaWQgc3R5bGUgcmVsZWFzZSBwcm9maWxlIikNCiAgICAgICAgbGlzdHMgPSB7a2V5OiBfc3RyaW5ncyhyb3dba2V5XSwga2V5LCByZXF1aXJlZD1rZXkgaW4gKCJ0YWdzIiwgInNlcnZpY2VzIiksDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbGltaXQ9NTAwIGlmIGtleSA9PSAiZXZpZGVuY2UiIGVsc2UgMjQwKQ0KICAgICAgICAgICAgICAgICBmb3Iga2V5IGluICgidGFncyIsICJzZXJ2aWNlcyIsICJpbnB1dF9jb25zdHJhaW50cyIsICJldmlkZW5jZSIpfQ0KICAgICAgICBpZiByb3dbImVuYWJsZWQiXSBhbmQgKHJvd1sicmV2aWV3X3N0YXR1cyJdICE9ICJhY2NlcHRlZCINCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBvciBwcm9maWxlIGlzIE5vbmUgb3Igbm90IGxpc3RzWyJldmlkZW5jZSJdKToNCiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkVuYWJsZWQgc3R5bGUgcmVxdWlyZXMgYW4gYWNjZXB0ZWQgcmV2aWV3ZWQgZXhlY3V0aW9uIHByb2ZpbGUiKQ0KICAgICAgICBmb3VuZC5hcHBlbmQoQ2FuZGlkYXRlKCoqeyoqcm93LCAqKmxpc3RzfSkpDQogICAgICAgIHNlZW4uYWRkKHJvd1siaWQiXSkNCiAgICByZXR1cm4gU3R5bGVSZWxlYXNlKHZhbHVlWyJyZWxlYXNlX2lkIl0sIGNhbm9uaWNhbF9kaWdlc3QodmFsdWUpLCB0dXBsZShmb3VuZCkpDQoNCg0KZGVmIGxvYWRfcmVsZWFzZShwYXRoOiBQYXRoID0gUEFUSCkgLT4gU3R5bGVSZWxlYXNlOg0KICAgIGlmIHBhdGguc3RhdCgpLnN0X3NpemUgPiA2NTUzNjoNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiU3R5bGUgcmVsZWFzZSBleGNlZWRzIHNpemUgbGltaXQiKQ0KICAgIHJldHVybiBwYXJzZV9yZWxlYXNlKGpzb24ubG9hZHMocGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IiksDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBvYmplY3RfcGFpcnNfaG9vaz1fdW5pcXVlX29iamVjdCkpDQoNCg0KUkVMRUFTRSA9IGxvYWRfcmVsZWFzZSgpDQo=', 'sha256': '5a28d367679b6997393a0c91f74f02cc7a0cdfe321a46e0e8505f599dc8dc347'}, 'scripts/hair_release_worker.py': {'data': 'IiIiSW5pdGlhbGl6YXRpb24gaG9vayBmb3IgdmVyaWZpZWQgYXBwcm92ZWQgSGFpciB3b3JrZXIgcmVsZWFzZXMuCgpJbnN0YWxsZWQgb25seSBpbiBhIGZyZXNoLCBoYXNoLXZlcmlmaWVkIHdvcmtlciB3b3Jrc3BhY2UuIE9yaWdpbmFsIHRyYWluaW5nCnJlZ2lzdHJpZXMsIGNoZWNrcG9pbnQgbWV0YWRhdGEgYW5kIG9yaWdpbmFsIGdlbmVyYXRpb24gbWV0aG9kcyBzdGF5IGludGFjdC4KIiIiCmltcG9ydCBiYXNlNjQKZnJvbSBpbyBpbXBvcnQgQnl0ZXNJTwppbXBvcnQgbWF0aAppbXBvcnQgdGltZQpmcm9tIFBJTCBpbXBvcnQgSW1hZ2UsIEltYWdlT3BzCgpmcm9tIGFwcC5oYWlyX3Byb2ZpbGVzIGltcG9ydCAoU1RZTEVfSURTLCBoYWlyX2NhbmRpZGF0ZXMsIGV4cGVjdGVkX2NvbnRyYWN0LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgd29ya2VyX2NhcGFiaWxpdGllcykKCgpkZWYgaW5zdGFsbChzZXJ2ZXIpOgogICAgaWYgZ2V0YXR0cihzZXJ2ZXIsICJfaGFpcl9yZWxlYXNlX2luc3RhbGxlZCIsIEZhbHNlKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkhhaXIgcmVsZWFzZSBob29rIGlzIGFscmVhZHkgaW5zdGFsbGVkIikKICAgIGlmIHNlcnZlci5ydW50aW1lLnJlYWR5IG9yIHNlcnZlci5ydW50aW1lLmJhc2VfbG9hZGVkIG9yIHNlcnZlci5ydW50aW1lLm93bmVyIGlzIG5vdCBOb25lOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiRG8gbm90IGNoYW5nZSBhIHJ1bm5pbmcgd29ya2VyIikKICAgIGFkZGl0aW9ucyA9IGhhaXJfY2FuZGlkYXRlcygpCiAgICBpZiBub3QgYWRkaXRpb25zOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkhhaXIgcmVsZWFzZSByZXF1aXJlcyBhcHByb3ZlZCBpbnN0YWxsZWQgSGFpciBwcm9maWxlcyIpCiAgICBlbmFibGVkX2lkcyA9IHtjLmlkIGZvciBjIGluIGFkZGl0aW9uc30KICAgIG9yaWdpbmFsX3J1bnRpbWUgPSBzZXJ2ZXIuRmx1eFJ1bnRpbWUKICAgIG9yaWdpbmFsX2NvbnRyYWN0ID0gc2VydmVyLmV4cGVjdGVkX2NvbnRyYWN0CiAgICBvcmlnaW5hbF91bmlmaWVkX3J1bnRpbWUgPSBzZXJ2ZXIuVW5pZmllZFJ1bnRpbWUKCiAgICBjbGFzcyBQcm9maWxlZEhhaXJSdW50aW1lKG9yaWdpbmFsX3J1bnRpbWUpOgogICAgICAgIGRlZiBnZW5lcmF0ZShzZWxmLCBpbWFnZSwgc3R5bGVfaWQpOgogICAgICAgICAgICBpZiBzdHlsZV9pZCBub3QgaW4gZW5hYmxlZF9pZHM6CiAgICAgICAgICAgICAgICByZXR1cm4gc3VwZXIoKS5nZW5lcmF0ZShpbWFnZSwgc3R5bGVfaWQpCiAgICAgICAgICAgIGNvbnRyYWN0ID0gZXhwZWN0ZWRfY29udHJhY3Qoc3R5bGVfaWQsIHNlbGYubWV0YWRhdGEpCiAgICAgICAgICAgIGlmIHNlbGYuYWN0aXZlX2FkYXB0ZXIgIT0gY29udHJhY3RbImFkYXB0ZXJfaWQiXSBvciBzZWxmLmFjdGl2ZV9hZGFwdGVyIG5vdCBpbiBzZWxmLmFkYXB0ZXJzOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJBcHByb3ZlZCBIYWlyIHJlcXVpcmVzIGl0cyB2ZXJpZmllZCBhY3RpdmUgYWRhcHRlciIpCiAgICAgICAgICAgIHBpcGUsIHRvcmNoID0gc2VsZi5waXBlLCBzZWxmLnRvcmNoCiAgICAgICAgICAgIGFjdGl2ZSA9IHBpcGUuZ2V0X2FjdGl2ZV9hZGFwdGVycygpCiAgICAgICAgICAgIGlmIGxlbihhY3RpdmUpICE9IDE6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkFwcHJvdmVkIEhhaXIgYWRhcHRlciBzdGF0ZSBpcyB1bmNlcnRhaW4iKQogICAgICAgICAgICBuYW1lID0gYWN0aXZlWzBdCiAgICAgICAgICAgIGRlZmF1bHRzID0ge30KICAgICAgICAgICAgZm9yIGtleSwgbW9kdWxlIGluIHBpcGUudHJhbnNmb3JtZXIubmFtZWRfbW9kdWxlcygpOgogICAgICAgICAgICAgICAgaWYgaGFzYXR0cihtb2R1bGUsICJsb3JhX0EiKToKICAgICAgICAgICAgICAgICAgICBpZiAoc2V0KG1vZHVsZS5sb3JhX0EpICE9IHtuYW1lfSBvciBzZXQobW9kdWxlLmxvcmFfQikgIT0ge25hbWV9CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBvciBtb2R1bGUuZGlzYWJsZV9hZGFwdGVycyBvciBtb2R1bGUubWVyZ2VkCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBvciBsaXN0KG1vZHVsZS5hY3RpdmVfYWRhcHRlcnMpICE9IFtuYW1lXSk6CiAgICAgICAgICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiQXBwcm92ZWQgSGFpciBMb1JBIGxheWVyIGlzIG5vdCBpc29sYXRlZCIpCiAgICAgICAgICAgICAgICAgICAgZGVmYXVsdHNba2V5XSA9IGZsb2F0KG1vZHVsZS5zY2FsaW5nW25hbWVdKQogICAgICAgICAgICBpZiBub3QgZGVmYXVsdHMgb3IgYW55KG5vdCBtYXRoLmlzZmluaXRlKHYpIG9yIHYgPD0gMCBmb3IgdiBpbiBkZWZhdWx0cy52YWx1ZXMoKSk6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkFwcHJvdmVkIEhhaXIgaGFzIG5vIHZhbGlkIExvUkEgc2NhbGluZyIpCiAgICAgICAgICAgIHBpcGUuc2V0X2FkYXB0ZXJzKG5hbWUsIGFkYXB0ZXJfd2VpZ2h0cz0xLjApCiAgICAgICAgICAgIGZvciBrZXksIG1vZHVsZSBpbiBwaXBlLnRyYW5zZm9ybWVyLm5hbWVkX21vZHVsZXMoKToKICAgICAgICAgICAgICAgIGlmIGtleSBpbiBkZWZhdWx0cyBhbmQgbm90IG1hdGguaXNjbG9zZShmbG9hdChtb2R1bGUuc2NhbGluZ1tuYW1lXSksIGRlZmF1bHRzW2tleV0sIHJlbF90b2w9MWUtNik6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJBcHByb3ZlZCBIYWlyIGFkYXB0ZXIgaW5mbHVlbmNlIGRpZmZlcnMgZnJvbSBmdWxsIGxvYWRlZCBzY2FsaW5nIikKICAgICAgICAgICAgc291cmNlID0gSW1hZ2VPcHMucGFkKGltYWdlLCAoNTEyLCA1MTIpLCBtZXRob2Q9SW1hZ2UuUmVzYW1wbGluZy5MQU5DWk9TLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY29sb3I9KDI0NSwgMjQ1LCAyNDUpLCBjZW50ZXJpbmc9KC41LCAuNSkpCiAgICAgICAgICAgIHRvcmNoLmN1ZGEucmVzZXRfcGVha19tZW1vcnlfc3RhdHMoMCkKICAgICAgICAgICAgc3RhcnRlZCA9IHRpbWUubW9ub3RvbmljKCkKICAgICAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgICAgICByZXN1bHQgPSBwaXBlKHByb21wdD1jb250cmFjdFsicHJvbXB0Il0sIGltYWdlPXNvdXJjZSwgd2lkdGg9NTEyLCBoZWlnaHQ9NTEyLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICBudW1faW5mZXJlbmNlX3N0ZXBzPTIwLCBndWlkYW5jZV9zY2FsZT00LjAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGdlbmVyYXRvcj10b3JjaC5HZW5lcmF0b3IoZGV2aWNlPSJjdWRhIikubWFudWFsX3NlZWQoMTk3NykpLmltYWdlc1swXS5jb252ZXJ0KCJSR0IiKQogICAgICAgICAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKDApCiAgICAgICAgICAgIGlmIHJlc3VsdC5zaXplICE9ICg1MTIsIDUxMik6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkFwcHJvdmVkIEhhaXIgb3V0cHV0IGRpbWVuc2lvbnMgZGlmZmVyIikKICAgICAgICAgICAgc3RyZWFtID0gQnl0ZXNJTygpCiAgICAgICAgICAgIHJlc3VsdC5zYXZlKHN0cmVhbSwgIlBORyIpCiAgICAgICAgICAgIHJldHVybiB7InN0YXR1cyI6ICJjb21wbGV0ZWQiLCAiZ2VuZXJhdG9yIjogY29udHJhY3RbImdlbmVyYXRvciJdLAogICAgICAgICAgICAgICAgICAgICJpbWFnZSI6IHsiZGF0YV91cmwiOiAiZGF0YTppbWFnZS9wbmc7YmFzZTY0LCIgKyBiYXNlNjQuYjY0ZW5jb2RlKHN0cmVhbS5nZXR2YWx1ZSgpKS5kZWNvZGUoKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImNvbnRlbnRfdHlwZSI6ICJpbWFnZS9wbmciLCAid2lkdGgiOiA1MTIsICJoZWlnaHQiOiA1MTJ9LAogICAgICAgICAgICAgICAgICAgICJtZXRhZGF0YSI6IHsqKntrOiB2IGZvciBrLCB2IGluIGNvbnRyYWN0Lml0ZW1zKCkgaWYgayAhPSAiZ2VuZXJhdG9yIn0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJydW50aW1lX3NlY29uZHMiOiByb3VuZCh0aW1lLm1vbm90b25pYygpIC0gc3RhcnRlZCwgMyksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJwZWFrX2dwdV9taWIiOiByb3VuZCh0b3JjaC5jdWRhLm1heF9tZW1vcnlfYWxsb2NhdGVkKDApIC8gMTAyNCoqMiwgMSl9fQoKICAgIGRlZiBjb250cmFjdChmZWF0dXJlLCBzdHlsZV9pZCwgYWRhcHRlcl9pZD1Ob25lLCBhZGFwdGVyX21ldGE9Tm9uZSk6CiAgICAgICAgaWYgZmVhdHVyZSA9PSAiaGFpcnN0eWxlIiBhbmQgc3R5bGVfaWQgaW4gZW5hYmxlZF9pZHM6CiAgICAgICAgICAgIGlmIGFkYXB0ZXJfaWQgIT0gZXhwZWN0ZWRfY29udHJhY3Qoc3R5bGVfaWQpWyJhZGFwdGVyX2lkIl06CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJBcHByb3ZlZCBIYWlyIHdhcyByb3V0ZWQgdG8gdGhlIHdyb25nIGFkYXB0ZXIiKQogICAgICAgICAgICByZXR1cm4gZXhwZWN0ZWRfY29udHJhY3Qoc3R5bGVfaWQsIGFkYXB0ZXJfbWV0YSkKICAgICAgICByZXR1cm4gb3JpZ2luYWxfY29udHJhY3QoZmVhdHVyZSwgc3R5bGVfaWQsIGFkYXB0ZXJfaWQsIGFkYXB0ZXJfbWV0YSkKCiAgICBjbGFzcyBSZWxlYXNlUnVudGltZShvcmlnaW5hbF91bmlmaWVkX3J1bnRpbWUpOgogICAgICAgIGRlZiBoZWFsdGgoc2VsZik6CiAgICAgICAgICAgIHJlcG9ydCA9IHN1cGVyKCkuaGVhbHRoKCkKICAgICAgICAgICAgcmVwb3J0WyJzdHlsZV9jYXBhYmlsaXRpZXMiXSA9IHdvcmtlcl9jYXBhYmlsaXRpZXMoewogICAgICAgICAgICAgICAgImhhaXJzdHlsZSI6IHNvcnRlZChzZXJ2ZXIuSEFJUl9TVFlMRVMpLAogICAgICAgICAgICAgICAgIm1ha2V1cCI6IHNvcnRlZChzZXJ2ZXIuTUFLRVVQX1BST01QVFMpLCAibmFpbHMiOiBzb3J0ZWQoc2VydmVyLk1PREVMX1NUWUxFUyksCiAgICAgICAgICAgIH0pCiAgICAgICAgICAgIHJldHVybiByZXBvcnQKCiAgICAjIFRoaXMgZGljdGlvbmFyeSBjb250cm9scyBwdWJsaWMgcm91dGluZyBvbmx5LiBSRUdJU1RSWSBhbmQgdGhlIGNoZWNrcG9pbnQncwogICAgIyBzdXBwb3J0ZWRfc3R5bGVfaWRzIHJlbWFpbiB1bmNoYW5nZWQgYW5kIHJldGFpbiB0aGVpciBvcmlnaW5hbCB2YWxpZGF0b3JzLgogICAgc2VydmVyLkhBSVJfU1RZTEVTID0geyoqc2VydmVyLkhBSVJfU1RZTEVTLCAqKntjLmlkOiB7CiAgICAgICAgImFkYXB0ZXJfaWQiOiBleHBlY3RlZF9jb250cmFjdChjLmlkKVsiYWRhcHRlcl9pZCJdLCAicHJvbXB0IjogZXhwZWN0ZWRfY29udHJhY3QoYy5pZClbInByb21wdCJdLAogICAgfSBmb3IgYyBpbiBhZGRpdGlvbnN9fQogICAgc2VydmVyLkZsdXhSdW50aW1lID0gUHJvZmlsZWRIYWlyUnVudGltZQogICAgc2VydmVyLmV4cGVjdGVkX2NvbnRyYWN0ID0gY29udHJhY3QKICAgIHNlcnZlci5VbmlmaWVkUnVudGltZSA9IFJlbGVhc2VSdW50aW1lCiAgICBzZXJ2ZXIucnVudGltZSA9IFJlbGVhc2VSdW50aW1lKCkKICAgIHNlcnZlci5faGFpcl9yZWxlYXNlX2luc3RhbGxlZCA9IFRydWUKICAgIHJldHVybiBzZXJ2ZXIK', 'sha256': 'db25604acedaeacdd6c554d917d0c5d3de9cab60dd737694cdbeee0fa3abfa5c'}, 'scripts/inference_performance.py': {'data': 'IiIiT3B0LWluIHBoYXNlIHRpbWluZyBhbmQgYm91bmRlZCBmaXhlZC1wcm9tcHQgcmV1c2U7IG5vIGltYWdlL3Jlc3VsdCBjYWNoZS4NCg0KRGVmYXVsdCBzZXJ2aW5nIGRvZXMgbm90IGluc3RhbGwgd3JhcHBlcnMuIEEgY2FuZGlkYXRlIG11c3QgcGFzcyBtYXRjaGVkIEdQVQ0KcXVhbGl0eS90aW1pbmcgcmV2aWV3IGJlZm9yZSBlbmFibGluZyBBSV9QUk9NUFRfQ0FDSEUgaW4gYSBkZXBsb3ltZW50Lg0KIiIiDQpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBPcmRlcmVkRGljdA0KZnJvbSBjb250ZXh0bGliIGltcG9ydCBjb250ZXh0bWFuYWdlcg0KZnJvbSBoYXNobGliIGltcG9ydCBzaGEyNTYNCmltcG9ydCBpbnNwZWN0DQppbXBvcnQganNvbg0KaW1wb3J0IHRpbWUNCg0KDQpjbGFzcyBJbmZlcmVuY2VQZXJmb3JtYW5jZToNCiAgICBkZWYgX19pbml0X18oc2VsZiwgc3luY2hyb25pemUsICosIGNhY2hlX2VuYWJsZWQ9RmFsc2UsIG1heF9lbnRyaWVzPTQsIG1heF9ieXRlcz02NCoxMDI0KjEwMjQsIHBoYXNlX3RpbWluZz1UcnVlLCBjbG9jaz10aW1lLm1vbm90b25pYywgY2FjaGVfZmVhdHVyZXM9Tm9uZSk6DQogICAgICAgIGlmIG5vdCAxIDw9IG1heF9lbnRyaWVzIDw9IDE2IG9yIG5vdCAxIDw9IG1heF9ieXRlcyA8PSAxMjgqMTAyNCoxMDI0Og0KICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignSW52YWxpZCBwcm9tcHQgY2FjaGUgYm91bmRzJykNCiAgICAgICAgc2VsZi5zeW5jaHJvbml6ZSwgc2VsZi5jbG9jayA9IHN5bmNocm9uaXplLCBjbG9jaw0KICAgICAgICBzZWxmLmNhY2hlX2VuYWJsZWQsIHNlbGYubWF4X2VudHJpZXMsIHNlbGYubWF4X2J5dGVzID0gY2FjaGVfZW5hYmxlZCwgbWF4X2VudHJpZXMsIG1heF9ieXRlcw0KICAgICAgICBzZWxmLmNhY2hlX2ZlYXR1cmVzID0gTm9uZSBpZiBjYWNoZV9mZWF0dXJlcyBpcyBOb25lIGVsc2UgZnJvemVuc2V0KGNhY2hlX2ZlYXR1cmVzKQ0KICAgICAgICBzZWxmLmNhY2hlID0gT3JkZXJlZERpY3QoKQ0KICAgICAgICBzZWxmLmNhY2hlX2J5dGVzID0gMA0KICAgICAgICBzZWxmLnBoYXNlX3RpbWluZyA9IHBoYXNlX3RpbWluZw0KICAgICAgICBzZWxmLmxhc3QgPSBOb25lDQogICAgICAgIHNlbGYuX21lYXN1cmluZyA9IEZhbHNlDQoNCiAgICBkZWYgY2FjaGVfZm9yKHNlbGYsIGZlYXR1cmUpOg0KICAgICAgICByZXR1cm4gc2VsZi5jYWNoZV9lbmFibGVkIGFuZCAoc2VsZi5jYWNoZV9mZWF0dXJlcyBpcyBOb25lIG9yIGZlYXR1cmUgaW4gc2VsZi5jYWNoZV9mZWF0dXJlcykNCg0KICAgIGRlZiBhcHBsaWVzX3RvKHNlbGYsIGZlYXR1cmUpOg0KICAgICAgICAjIEhhaXIvTWFrZXVwIGJ5cGFzcyB0aGUgd3JhcHBlciBhbmQgaXRzIHN5bmNocm9uaXphdGlvbnMgZW50aXJlbHkgaW4gTmFpbHMtb25seSBzZXJ2aW5nLg0KICAgICAgICByZXR1cm4gc2VsZi5waGFzZV90aW1pbmcgb3Igc2VsZi5jYWNoZV9mb3IoZmVhdHVyZSkNCg0KICAgIGRlZiBjbGVhcihzZWxmKToNCiAgICAgICAgc2VsZi5jYWNoZS5jbGVhcigpDQogICAgICAgIHNlbGYuY2FjaGVfYnl0ZXMgPSAwDQoNCiAgICBAc3RhdGljbWV0aG9kDQogICAgZGVmIF9ieXRlcyh0ZW5zb3JzKToNCiAgICAgICAgcmV0dXJuIHN1bSh2YWx1ZS5udW1lbCgpKnZhbHVlLmVsZW1lbnRfc2l6ZSgpIGZvciB2YWx1ZSBpbiB0ZW5zb3JzKQ0KDQogICAgZGVmIF9rZXkoc2VsZiwgc2lnbmF0dXJlLCBhcmdzLCBrd2FyZ3MsIGlkZW50aXR5LCBhbGxvd2VkKToNCiAgICAgICAgYm91bmQgPSBzaWduYXR1cmUuYmluZCgqYXJncywgKiprd2FyZ3MpDQogICAgICAgIGJvdW5kLmFwcGx5X2RlZmF1bHRzKCkNCiAgICAgICAgdmFsdWVzID0gYm91bmQuYXJndW1lbnRzDQogICAgICAgIHByb21wdCA9IHZhbHVlcy5nZXQoJ3Byb21wdCcpDQogICAgICAgIGlmIHZhbHVlcy5nZXQoJ3Byb21wdF9lbWJlZHMnKSBpcyBub3QgTm9uZSBvciBub3QgaXNpbnN0YW5jZShwcm9tcHQsIHN0cikgb3IgcHJvbXB0IG5vdCBpbiBhbGxvd2VkOg0KICAgICAgICAgICAgcmV0dXJuIE5vbmUNCiAgICAgICAgIyBPbmx5IGRlY2xhcmVkIHB1YmxpYyBwcm9tcHRzIGNhbiBiZSBjYWNoZWQ7IGltYWdlL3VzZXIgYXJndW1lbnRzIG5ldmVyIGVudGVyIGEga2V5Lg0KICAgICAgICBpZiBzZXQodmFsdWVzKSAtIHsncHJvbXB0JywnZGV2aWNlJywnbnVtX2ltYWdlc19wZXJfcHJvbXB0JywncHJvbXB0X2VtYmVkcycsJ21heF9zZXF1ZW5jZV9sZW5ndGgnLCd0ZXh0X2VuY29kZXJfb3V0X2xheWVycyd9Og0KICAgICAgICAgICAgcmV0dXJuIE5vbmUNCiAgICAgICAgb3B0aW9ucyA9IHtrZXk6c3RyKHZhbHVlKSBmb3Iga2V5LHZhbHVlIGluIHZhbHVlcy5pdGVtcygpIGlmIGtleSBub3QgaW4gKCdwcm9tcHQnLCdwcm9tcHRfZW1iZWRzJyl9DQogICAgICAgIHJldHVybiAoaWRlbnRpdHksc2hhMjU2KHByb21wdC5lbmNvZGUoKSkuaGV4ZGlnZXN0KCksanNvbi5kdW1wcyhvcHRpb25zLHNvcnRfa2V5cz1UcnVlKSkNCg0KICAgIEBjb250ZXh0bWFuYWdlcg0KICAgIGRlZiBtZWFzdXJlKHNlbGYsIHBpcGUsICosIGlkZW50aXR5LCBhbGxvd2VkX3Byb21wdHMsIGZlYXR1cmU9Tm9uZSk6DQogICAgICAgIGlmIHNlbGYuX21lYXN1cmluZzoNCiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignQ29uY3VycmVudCBwZXJmb3JtYW5jZSBzY29wZSByZWZ1c2VkJykNCiAgICAgICAgc2VsZi5fbWVhc3VyaW5nID0gVHJ1ZQ0KICAgICAgICBwaGFzZV9zZWNvbmRzLCBjb3VudHMgPSB7fSwge30NCiAgICAgICAgc3RhdHMgPSB7J2hpdHMnOjAsJ21pc3Nlcyc6MCwnYnlwYXNzZXMnOjB9DQogICAgICAgIHJlc3RvcmUgPSBbXQ0KICAgICAgICBtYXJrZXIgPSBvYmplY3QoKQ0KICAgICAgICBzZWxmLmxhc3QgPSBOb25lDQoNCiAgICAgICAgZGVmIHdyYXBwZWQodGFyZ2V0LCBuYW1lLCBwaGFzZSwgcHJvbXB0PUZhbHNlKToNCiAgICAgICAgICAgIGlmIChub3Qgc2VsZi5waGFzZV90aW1pbmcgYW5kIG5vdCBwcm9tcHQpIG9yIG5vdCBoYXNhdHRyKHRhcmdldCxuYW1lKToNCiAgICAgICAgICAgICAgICByZXR1cm4NCiAgICAgICAgICAgIG9yaWdpbmFsID0gZ2V0YXR0cih0YXJnZXQsbmFtZSkNCiAgICAgICAgICAgIHByZXZpb3VzID0gdmFycyh0YXJnZXQpLmdldChuYW1lLG1hcmtlcikNCiAgICAgICAgICAgIHNpZ25hdHVyZSA9IGluc3BlY3Quc2lnbmF0dXJlKG9yaWdpbmFsKSBpZiBwcm9tcHQgZWxzZSBOb25lDQogICAgICAgICAgICBkZWYgaW52b2tlKCphcmdzLCoqa3dhcmdzKToNCiAgICAgICAgICAgICAgICBpZiBzZWxmLnBoYXNlX3RpbWluZzoNCiAgICAgICAgICAgICAgICAgICAgc2VsZi5zeW5jaHJvbml6ZSgpDQogICAgICAgICAgICAgICAgc3RhcnRlZCA9IHNlbGYuY2xvY2soKQ0KICAgICAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAgICAgICAgICAgaWYgbm90IHByb21wdCBvciBub3Qgc2VsZi5jYWNoZV9mb3IoZmVhdHVyZSk6DQogICAgICAgICAgICAgICAgICAgICAgICByZXR1cm4gb3JpZ2luYWwoKmFyZ3MsKiprd2FyZ3MpDQogICAgICAgICAgICAgICAgICAgIGtleSA9IHNlbGYuX2tleShzaWduYXR1cmUsYXJncyxrd2FyZ3MsKGlkZW50aXR5LHN0cihnZXRhdHRyKHBpcGUsJ2R0eXBlJyxOb25lKSksc3RyKGdldGF0dHIoZ2V0YXR0cihwaXBlLCd0ZXh0X2VuY29kZXInLE5vbmUpLCdkdHlwZScsTm9uZSkpLGlkKGdldGF0dHIocGlwZSwndGV4dF9lbmNvZGVyJyxwaXBlKSksc3RyKGdldGF0dHIocGlwZSwnX2V4ZWN1dGlvbl9kZXZpY2UnLE5vbmUpKSksYWxsb3dlZF9wcm9tcHRzKQ0KICAgICAgICAgICAgICAgICAgICBpZiBrZXkgaXMgTm9uZToNCiAgICAgICAgICAgICAgICAgICAgICAgIHN0YXRzWydieXBhc3NlcyddICs9IDENCiAgICAgICAgICAgICAgICAgICAgICAgIHJldHVybiBvcmlnaW5hbCgqYXJncywqKmt3YXJncykNCiAgICAgICAgICAgICAgICAgICAgaWYga2V5IGluIHNlbGYuY2FjaGU6DQogICAgICAgICAgICAgICAgICAgICAgICBjcHUsZGV2aWNlcyxzaXplID0gc2VsZi5jYWNoZS5wb3Aoa2V5KQ0KICAgICAgICAgICAgICAgICAgICAgICAgc2VsZi5jYWNoZVtrZXldID0gKGNwdSxkZXZpY2VzLHNpemUpDQogICAgICAgICAgICAgICAgICAgICAgICBzdGF0c1snaGl0cyddICs9IDENCiAgICAgICAgICAgICAgICAgICAgICAgIHJldHVybiB0dXBsZSh2YWx1ZS50byhkZXZpY2U9ZGV2aWNlKS5jbG9uZSgpIGZvciB2YWx1ZSxkZXZpY2UgaW4gemlwKGNwdSxkZXZpY2VzKSkNCiAgICAgICAgICAgICAgICAgICAgc3RhdHNbJ21pc3NlcyddICs9IDENCiAgICAgICAgICAgICAgICAgICAgdmFsdWUgPSBvcmlnaW5hbCgqYXJncywqKmt3YXJncykNCiAgICAgICAgICAgICAgICAgICAgIyBlbmNvZGVfcHJvbXB0IHJldHVybnMgdGhlIHR3byB0ZW5zb3JzIChlbWJlZGRpbmdzLCB0ZXh0IElEcykuDQogICAgICAgICAgICAgICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHZhbHVlLHR1cGxlKSBvciBsZW4odmFsdWUpIT0yIG9yIG5vdCBhbGwoaGFzYXR0cih2LCdkZXRhY2gnKSBmb3IgdiBpbiB2YWx1ZSk6DQogICAgICAgICAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ1Vuc3VwcG9ydGVkIHByb21wdCBlbmNvZGVyIHJldHVybiBjb250cmFjdCcpDQogICAgICAgICAgICAgICAgICAgIHNpemUgPSBzZWxmLl9ieXRlcyh2YWx1ZSkNCiAgICAgICAgICAgICAgICAgICAgaWYgc2l6ZSA8PSBzZWxmLm1heF9ieXRlczoNCiAgICAgICAgICAgICAgICAgICAgICAgIGNwdSA9IHR1cGxlKHYuZGV0YWNoKCkudG8oZGV2aWNlPSdjcHUnKS5jbG9uZSgpIGZvciB2IGluIHZhbHVlKQ0KICAgICAgICAgICAgICAgICAgICAgICAgd2hpbGUgc2VsZi5jYWNoZSBhbmQgKGxlbihzZWxmLmNhY2hlKT49c2VsZi5tYXhfZW50cmllcyBvciBzZWxmLmNhY2hlX2J5dGVzK3NpemU+c2VsZi5tYXhfYnl0ZXMpOg0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIF8sKF8sXyxyZW1vdmVkKT1zZWxmLmNhY2hlLnBvcGl0ZW0obGFzdD1GYWxzZSkNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBzZWxmLmNhY2hlX2J5dGVzIC09IHJlbW92ZWQNCiAgICAgICAgICAgICAgICAgICAgICAgIHNlbGYuY2FjaGVba2V5XT0oY3B1LHR1cGxlKHYuZGV2aWNlIGZvciB2IGluIHZhbHVlKSxzaXplKQ0KICAgICAgICAgICAgICAgICAgICAgICAgc2VsZi5jYWNoZV9ieXRlcyArPSBzaXplDQogICAgICAgICAgICAgICAgICAgIHJldHVybiB2YWx1ZQ0KICAgICAgICAgICAgICAgIGZpbmFsbHk6DQogICAgICAgICAgICAgICAgICAgIGlmIHNlbGYucGhhc2VfdGltaW5nOg0KICAgICAgICAgICAgICAgICAgICAgICAgc2VsZi5zeW5jaHJvbml6ZSgpDQogICAgICAgICAgICAgICAgICAgICAgICBwaGFzZV9zZWNvbmRzW3BoYXNlXSA9IHBoYXNlX3NlY29uZHMuZ2V0KHBoYXNlLDApKyhzZWxmLmNsb2NrKCktc3RhcnRlZCkNCiAgICAgICAgICAgICAgICAgICAgICAgIGNvdW50c1twaGFzZV0gPSBjb3VudHMuZ2V0KHBoYXNlLDApKzENCiAgICAgICAgICAgIHNldGF0dHIodGFyZ2V0LG5hbWUsaW52b2tlKQ0KICAgICAgICAgICAgcmVzdG9yZS5hcHBlbmQoKHRhcmdldCxuYW1lLHByZXZpb3VzKSkNCg0KICAgICAgICB0cnk6DQogICAgICAgICAgICBzZWxmLnN5bmNocm9uaXplKCkNCiAgICAgICAgICAgIHN0YXJ0ZWQgPSBzZWxmLmNsb2NrKCkNCiAgICAgICAgICAgIHdyYXBwZWQocGlwZSwnZW5jb2RlX3Byb21wdCcsJ3Byb21wdF9lbmNvZGluZycsVHJ1ZSkNCiAgICAgICAgICAgIHdyYXBwZWQocGlwZSwncHJlcGFyZV9pbWFnZV9sYXRlbnRzJywnaW1hZ2VfZW5jb2RpbmcnKQ0KICAgICAgICAgICAgd3JhcHBlZChwaXBlLnRyYW5zZm9ybWVyLCdmb3J3YXJkJywnZGVub2lzaW5nJykNCiAgICAgICAgICAgIHdyYXBwZWQocGlwZS52YWUsJ2RlY29kZScsJ2ltYWdlX2RlY29kaW5nJykNCiAgICAgICAgICAgIHlpZWxkDQogICAgICAgICAgICBzZWxmLnN5bmNocm9uaXplKCkNCiAgICAgICAgICAgIHNlbGYubGFzdCA9IHsncGhhc2Vfc2Vjb25kcyc6e2tleTpyb3VuZCh2YWx1ZSw2KSBmb3Iga2V5LHZhbHVlIGluIHBoYXNlX3NlY29uZHMuaXRlbXMoKX0sDQogICAgICAgICAgICAgICAgICAgICAgICAgJ3BoYXNlX2NhbGxzJzpjb3VudHMsJ3J1bnRpbWVfc2Vjb25kcyc6cm91bmQoc2VsZi5jbG9jaygpLXN0YXJ0ZWQsNiksDQogICAgICAgICAgICAgICAgICAgICAgICAgJ3Byb21wdF9jYWNoZSc6eyoqc3RhdHMsJ2VuYWJsZWQnOnNlbGYuY2FjaGVfZm9yKGZlYXR1cmUpLCdlbnRyaWVzJzpsZW4oc2VsZi5jYWNoZSksJ2NwdV9ieXRlcyc6c2VsZi5jYWNoZV9ieXRlc30sDQogICAgICAgICAgICAgICAgICAgICAgICAgJ3RpbWluZ19raW5kJzonc3luY2hyb25pemVkX3BoYXNlX2RpYWdub3N0aWMnIGlmIHNlbGYucGhhc2VfdGltaW5nIGVsc2UgJ3dob2xlX3J1bnRpbWVfc3luY2hyb25pemVkX25vX3BoYXNlX2JhcnJpZXJzJ30NCiAgICAgICAgZXhjZXB0IEJhc2VFeGNlcHRpb246DQogICAgICAgICAgICBzZWxmLmNsZWFyKCkNCiAgICAgICAgICAgIHJhaXNlDQogICAgICAgIGZpbmFsbHk6DQogICAgICAgICAgICBmb3IgdGFyZ2V0LG5hbWUscHJldmlvdXMgaW4gcmV2ZXJzZWQocmVzdG9yZSk6DQogICAgICAgICAgICAgICAgaWYgcHJldmlvdXMgaXMgbWFya2VyOg0KICAgICAgICAgICAgICAgICAgICBkZWxhdHRyKHRhcmdldCxuYW1lKQ0KICAgICAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgICAgIHNldGF0dHIodGFyZ2V0LG5hbWUscHJldmlvdXMpDQogICAgICAgICAgICBzZWxmLl9tZWFzdXJpbmc9RmFsc2UNCg0KDQpkZWYgY29uZmlndXJlZF9wZXJmb3JtYW5jZShzeW5jaHJvbml6ZSwgZW52KToNCiAgICBpZiBlbnYuZ2V0KCdBSV9JTkZFUkVOQ0VfUFJPRklMRScpIT0nMScgYW5kIGVudi5nZXQoJ0FJX1BST01QVF9DQUNIRScpIT0nMSc6DQogICAgICAgIHJldHVybiBOb25lDQogICAgc2NvcGUgPSBlbnYuZ2V0KCdBSV9QUk9NUFRfQ0FDSEVfU0NPUEUnLCAnbmFpbHMnKQ0KICAgIGlmIHNjb3BlIG5vdCBpbiAoJ25haWxzJywgJ2FsbCcpOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdVbnN1cHBvcnRlZCBwcm9tcHQgY2FjaGUgc2NvcGUnKQ0KICAgIHJldHVybiBJbmZlcmVuY2VQZXJmb3JtYW5jZShzeW5jaHJvbml6ZSxjYWNoZV9lbmFibGVkPWVudi5nZXQoJ0FJX1BST01QVF9DQUNIRScpPT0nMScscGhhc2VfdGltaW5nPWVudi5nZXQoJ0FJX0lORkVSRU5DRV9QUk9GSUxFJyk9PScxJyxjYWNoZV9mZWF0dXJlcz17J25haWxzJ30gaWYgc2NvcGU9PSduYWlscycgZWxzZSBOb25lKQ0K', 'sha256': 'b69dd604baa6a09ba1a1d7598e979c66a81d26f70a8e9ae3db3bb790766030af'}, 'scripts/unified_gate3_server.py': {'data': '...', 'sha256': '7a8cd5aa4ed57cceb6409d92cc08e8a867802c358db944f20e8c1f3d3b75c028'}}
stage_overlay(CLIENT_WORK, OPTIMIZED_SOURCES)
(CLIENT_WORK/'worker_overlay.py').write_text('"""Stage one matched Hair/Nails worker release in a fresh verified GPU workspace."""\nimport base64\nfrom hashlib import sha256\nimport json\nfrom pathlib import Path\nimport socket\n\nFILES = frozenset({\n    "scripts/unified_gate3_server.py", "scripts/inference_performance.py", "scripts/hair_release_worker.py",\n    "backend/app/hair_profiles.py", "backend/app/hair_profile.json",\n    "backend/app/hair_profile_butterfly_cut.json", "backend/app/hair_profile_wolf_cut.json",\n    "backend/app/hair_profile_korean_two_block.json", "backend/app/hair_profile_comma_hair.json",\n    "backend/app/style_release.py", "backend/app/style_release.json",\n})\n\n\ndef validated(payload):\n    if not isinstance(payload, dict) or set(payload) != FILES:\n        raise ValueError("Unexpected Hair/Nails worker source inventory")\n    sources = {}\n    for name, item in payload.items():\n        if not isinstance(item, dict) or set(item) != {"data", "sha256"}:\n            raise ValueError("Invalid release source record")\n        raw = base64.b64decode(item["data"], validate=True)\n        if sha256(raw).hexdigest() != item["sha256"]:\n            raise ValueError("Release source hash differs: " + name)\n        if name.endswith(".py"):\n            compile(raw, name, "exec")\n        else:\n            json.loads(raw)\n        sources[name] = raw\n    return sources\n\n\ndef stage_overlay(work, payload):\n    validated(payload)\n    target = Path(work) / "optimized-worker.json"\n    with target.open("x", encoding="utf-8") as file:\n        json.dump(payload, file)\n    return target\n\n\ndef install_overlay(root, payload_file, output):\n    root, output = Path(root), Path(output)\n    for port in (8765, 8800):\n        with socket.socket() as connection:\n            connection.settimeout(1)\n            if connection.connect_ex(("127.0.0.1", port)) == 0:\n                raise RuntimeError("Do not change a running AI service")\n    record = output / "hair-nails-worker-release.json"\n    if record.exists():\n        raise RuntimeError("Worker release already installed; use a fresh session")\n    sources = validated(json.loads(Path(payload_file).read_text(encoding="utf-8")))\n    manifest_path = root / "gate3_bundle.json"\n    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n    # Full original bundle verification is also required by the startup hook.\n    # Never rewrite the original manifest or any model/registry/checkpoint.\n    for name in FILES:\n        target = root / name\n        if name in manifest["files"]:\n            if sha256(target.read_bytes()).hexdigest() != manifest["files"][name]:\n                raise ValueError("Original worker source changed before release install")\n        elif target.exists():\n            raise ValueError("Unexpected preexisting worker extension: " + name)\n    for name, raw in sources.items():\n        target = root / name\n        target.parent.mkdir(parents=True, exist_ok=True)\n        target.write_bytes(raw)\n    evidence = {"schema": "beautycore-hair-nails-worker-v1",\n                "original_manifest_sha256": sha256(manifest_path.read_bytes()).hexdigest(),\n                "source_sha256": {name: sha256(raw).hexdigest() for name, raw in sources.items()},\n                "model_parameters_changed": False,\n                "hair_profile": "hair-expanded-v1", "optimization": "nails-only"}\n    record.write_bytes((json.dumps(evidence, indent=2) + "\\n").encode())\n    print("WORKER RELEASE: approved Hair profiles and existing Nails-only optimization.", flush=True)\n    return evidence\n', encoding='utf-8')
print('Approved Nail release verified. Run Cell 2 once.')

cpu_manifest = json.loads((CLIENT_WORK/'cpu-src/demo_bundle.json').read_text(encoding='utf-8'))
if cpu_manifest.get('worker_sources') != {name: row['sha256'] for name, row in OPTIMIZED_SOURCES.items()}:
    raise RuntimeError('CPU and worker release source pair differs')
print('Five Hair + Nails release staged. Run Cell 2 once.')


In [ ]:
import os
BEAUTYCORE_PUBLIC_URL = 'https://beautycore-demo.onrender.com'
os.environ['BEAUTYCORE_PUBLIC_URL'] = BEAUTYCORE_PUBLIC_URL
(CLIENT_WORK/'runtime_registration.py').write_text('"""HMAC metadata registration/heartbeat. No credentials or generation requests in logs."""\nfrom hashlib import sha256\nimport hmac\nimport json\nimport os\nfrom pathlib import Path\nimport socket\nimport sys\nimport time\nfrom urllib.error import HTTPError, URLError\nfrom urllib.parse import urlsplit\nfrom urllib.request import Request, HTTPRedirectHandler, build_opener\n\nSCHEMA = "beautycore-runtime-v1"\nHEARTBEAT_SECONDS = 60\n\nclass RegistrationError(RuntimeError):\n    def __init__(self, status=None):\n        self.status = status\n        super().__init__("Website connection not acknowledged" + (f" (HTTP {status})" if status else ""))\n\nclass NoRedirect(HTTPRedirectHandler):\n    def redirect_request(self, req, fp, code, msg, headers, newurl):\n        return None\n\ndef canonical(message):\n    return json.dumps(message, sort_keys=True, separators=(",", ":"), ensure_ascii=True).encode()\n\ndef sign(message, key):\n    derived = hmac.new(key.encode(), b"beautycore-runtime-key-v1", sha256).digest()\n    return hmac.new(derived, canonical(message), sha256).hexdigest()\n\ndef website_root(website):\n    p = urlsplit(website)\n    if (p.scheme != "https" or not p.hostname or p.username or p.password or p.port\n            or p.query or p.fragment or p.path.rstrip("/")):\n        raise RegistrationError()\n    return website.rstrip("/")\n\ndef send(website, record, key, operation, *, opener=None, now=time.time):\n    if len(key) < 32:\n        raise RegistrationError()\n    message = {"schema": SCHEMA, "operation": operation, "session_id": record["session_id"],\n               "endpoint": record["url"], "started_at": record["started_at"], "issued_at": int(now())}\n    request = Request(website_root(website) + "/api/internal/ai-runtime/register", data=canonical(message),\n                      headers={"Content-Type": "application/json", "x-ai-runtime-signature": sign(message, key)})\n    try:\n        with (opener or build_opener(NoRedirect())).open(request, timeout=75) as response:\n            if response.status != 200:\n                raise RegistrationError(response.status)\n            raw = response.read(8193)\n            if len(raw) > 8192:\n                raise RegistrationError()\n            ack = json.loads(raw)\n            if (ack.get("connected") is not True or ack.get("session_id") != record["session_id"]\n                    or not isinstance(ack.get("expires_at"), (int, float)) or ack["expires_at"] <= now() * 1000):\n                raise RegistrationError()\n            return ack\n    except HTTPError as error:\n        raise RegistrationError(error.code) from None\n    except (URLError, OSError, TimeoutError, ValueError, TypeError, AttributeError):\n        raise RegistrationError() from None\n\ndef register(website, record, key, *, send_fn=send, sleep=time.sleep, attempts=4):\n    # Metadata only. The GPU request is never repeated here. Accommodates Render cold start.\n    for attempt in range(attempts):\n        try:\n            return send_fn(website, record, key, "register")\n        except RegistrationError as error:\n            if error.status in (400, 401, 403, 409, 404) or attempt == attempts - 1:\n                raise\n            print("Website registration waiting; AI processes remain running.", flush=True)\n            sleep(10)\n\ndef services_alive(record):\n    try:\n        for field in ("api_pid", "tunnel_pid"):\n            os.kill(record[field], 0)\n        for port in (8765, 8800):\n            with socket.socket() as connection:\n                connection.settimeout(2)\n                if connection.connect_ex(("127.0.0.1", port)) != 0:\n                    return False\n        return True\n    except (OSError, KeyError, TypeError):\n        return False\n\ndef heartbeat(website, record, key, *, send_fn=send, alive=services_alive, sleep=time.sleep, now=time.time):\n    acknowledged_until = record["registration_expires_at"]\n    while True:\n        sleep(HEARTBEAT_SECONDS)\n        if not alive(record):\n            print("AI process stopped. Website lease will expire; start a fresh notebook session.", flush=True)\n            return\n        try:\n            ack = send_fn(website, record, key, "renew")\n            acknowledged_until = ack["expires_at"]\n        except RegistrationError as error:\n            if error.status in (400, 401, 403, 409, 404) or now() * 1000 >= acknowledged_until:\n                print("Website lease expired or refused. Start a fresh session; do not replay generation.", flush=True)\n                return\n            print("Website heartbeat not acknowledged; existing lease has not been extended.", flush=True)\n\nif __name__ == "__main__":\n    record = json.loads(Path(sys.argv[2]).read_text())\n    heartbeat(sys.argv[1], record, os.environ.get("AI_BACKEND_API_KEY", ""))\n', encoding='utf-8')
"""New scheduled-demo runner. Existing notebooks and GPU bundle are not modified."""
import argparse
import json
import os
from pathlib import Path
import re
import socket
import subprocess
import sys
import time
from uuid import uuid4
from urllib.request import Request, urlopen

WORK = CLIENT_WORK
GPU_ROOT = WORK / "gpu-src"
CPU_ROOT = WORK / "cpu-src"
PORT = 8800


def command(args, cwd, env, stage):
    print("DEMO STAGE:", stage, flush=True)
    log = WORK / (stage + ".log")
    with log.open("w") as output:
        process = subprocess.Popen(args, cwd=cwd, env=env, stdout=output, stderr=subprocess.STDOUT)
        while process.poll() is None:
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                print(stage, "still running; log bytes:", log.stat().st_size, flush=True)
    if process.returncode:
        raise RuntimeError(stage + " failed; inspect its log")


def read_json(url, key=None):
    headers = {"x-ai-backend-key": key, "x-ai-user-id": "demo-operator"} if key else {}
    with urlopen(Request(url, headers=headers), timeout=40) as response:
        return json.load(response)


def main():
    if sys.platform != "linux":
        raise RuntimeError("Run in the new Kaggle Linux GPU notebook")
    if (WORK / "demo-session.json").exists():
        raise RuntimeError("Refuse duplicate startup; use a fresh notebook session")
    for port in (8765, PORT):
        with socket.socket() as connection:
            connection.settimeout(1)
            if connection.connect_ex(("127.0.0.1", port)) == 0:
                raise RuntimeError("An AI service is already running; do not start another worker")
    # Total RAM, not VRAM. Old GPU evidence peaked near 23 GiB before CPU vision.
    memory = int(re.search(r"MemTotal:\s+(\d+)", Path("/proc/meminfo").read_text())[1]) * 1024
    if memory < 30 * 1024**3:
        raise RuntimeError("At least 30 GiB host RAM required for this unverified combined workload")
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    backend_key = secrets.get_secret("AI_BACKEND_API_KEY") or ""
    gpu_key = secrets.get_secret("AI_REMOTE_API_KEY") or ""
    gemini_key = secrets.get_secret("GEMINI_API_KEY") or ""
    if len(backend_key) < 32 or len(gpu_key) < 24 or not gemini_key:
        raise RuntimeError("Enable the three required Kaggle Secrets; never put keys into cells")
    session_id = str(uuid4())
    started_at = int(time.time())
    # Reserve the session before starting any subprocess; failed sessions need a fresh restart.
    (WORK / "demo-session.json").write_text(json.dumps({"status": "starting", "host_ram_bytes": memory}))
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    # CPU services do not use Kaggle's inline-notebook plotting backend.
    cpu_env = {**env, "CUDA_VISIBLE_DEVICES": "", "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "MPLBACKEND": "Agg"}
    import ctypes.util
    if not ctypes.util.find_library("portaudio"):
        command(["apt-get", "update"], CPU_ROOT, env, "portaudio-index")
        command(["apt-get", "install", "-y", "libportaudio2"], CPU_ROOT, env, "portaudio-install")
    for name in ("backend-env", "segment-env"):
        # Kaggle ensurepip can fail even on the reviewed Python image.
        # Use host pip only as an installer targeting the isolated CPU environment.
        command([sys.executable, "-m", "venv", "--without-pip", str(WORK / name)], CPU_ROOT, cpu_env, name + "-create")
        command([sys.executable, "-m", "pip", "--python", str(WORK / name), "install", "pip==25.3"],
                CPU_ROOT, cpu_env, name + "-pip-bootstrap")
    api_python = WORK / "backend-env/bin/python"
    seg_python = WORK / "segment-env/bin/python"
    command([str(api_python), "-m", "pip", "install", "-r", "backend/requirements-nails.txt", "--constraint", "backend/requirements-demo-constraints.txt"],
            CPU_ROOT, cpu_env, "backend-install")
    command([str(seg_python), "-m", "pip", "install", "torch==2.10.0", "torchvision==0.25.0", "--index-url", "https://download.pytorch.org/whl/cpu"],
            CPU_ROOT, cpu_env, "segment-torch-install")
    command([str(seg_python), "-m", "pip", "install", "-r", "backend/requirements-nails-segmenter.txt"],
            CPU_ROOT, cpu_env, "segment-install")
    for name, interpreter in (("backend", api_python), ("segment", seg_python)):
        command([str(interpreter), "-m", "pip", "check"], CPU_ROOT, cpu_env, name + "-pip-check")
    checkpoint = CPU_ROOT / "data/nails/checkpoints/mnemic/nails_seg_s_yolov8_v1.pt"
    command([str(seg_python), "-c", "import torch, torchvision; from ultralytics import YOLO; assert torchvision.extension._has_ops(); assert '+cpu' in torch.__version__; YOLO(" + repr(str(checkpoint)) + ")"],
            CPU_ROOT, cpu_env, "segment-load")
    landmark = CPU_ROOT / "data/nails/checkpoints/mediapipe/hand_landmarker.task"
    cpu_env["PYTHONPATH"] = str(CPU_ROOT / "backend")
    command([str(api_python), "-c", "from pathlib import Path; from app.nails.geometry import MediaPipeHandLocalizer; MediaPipeHandLocalizer(Path(" + repr(str(landmark)) + "))"],
            CPU_ROOT, cpu_env, "landmark-load")
    gpu_code = f"""from pathlib import Path
from scripts import unified_gate3_bootstrap as original
from scripts.capstone_train002_bootstrap import worker_environment
start = original.start_server
import importlib.util
spec = importlib.util.spec_from_file_location('optimized_overlay', Path({str(WORK / 'worker_overlay.py')!r}))
release_overlay = importlib.util.module_from_spec(spec)
spec.loader.exec_module(release_overlay)

def start_optimized(output, env):
    original.verify_bundle()
    release_overlay.install_overlay(original.ROOT, Path({str(WORK / 'optimized-worker.json')!r}), output)
    return start(output, worker_environment(dict(env, AI_PROMPT_CACHE='1', AI_PROMPT_CACHE_SCOPE='nails', AI_INFERENCE_PROFILE='0')))
original.start_server = start_optimized
report = original.bootstrap(Path({str(WORK / 'gpu-session')!r}), local_only=True)
assert report['status'] == 'READY_FOR_LIVE_ACCEPTANCE', 'GPU bootstrap failed'
"""
    # Preserves exact host/package/model/adapter checks; one GPU load, no public GPU tunnel.
    command([sys.executable, "-c", gpu_code], GPU_ROOT,
            {**env, "AI_REMOTE_API_KEY": gpu_key}, "gpu-bootstrap")
    cpu_env.update(AI_BACKEND_API_KEY=backend_key, AI_GPU_LOOPBACK="1",
                   AI_REMOTE_URL="http://127.0.0.1:8765", AI_REMOTE_API_KEY=gpu_key,
                   GENERATION_ENGINE="remote_flux", MAKEUP_GENERATION_ENGINE="remote_makeup",
                   NAILS_PREVIEW_MODE="hybrid", NAILS_INFERENCE_STEPS="8",
                   NAILS_HAND_LANDMARKER_PATH=str(landmark), NAILS_SEGMENT_CHECKPOINT=str(checkpoint),
                   NAILS_SEGMENT_PYTHON=str(seg_python), CONSULTATION_PROVIDER="gemini",
                   CONSULTATION_GEMINI_MODEL="gemini-3.5-flash-lite", GEMINI_API_KEY=gemini_key,
                   HAIRCAPSTONE_STYLE_REGISTRY_PATH=str(CPU_ROOT / "backend/app/style_registry_train002_smoke.json"),
                   FRONTEND_ORIGINS="")
    with (WORK / "api.log").open("w") as log:
        api = subprocess.Popen([str(api_python), "-m", "uvicorn", "app.demo_server:app",
                                "--host", "127.0.0.1", "--port", str(PORT), "--workers", "1"],
                               cwd=CPU_ROOT, env=cpu_env, stdout=log, stderr=subprocess.STDOUT,
                               start_new_session=True)
    deadline = time.monotonic() + 120
    while True:
        if api.poll() is not None:
            raise RuntimeError("API exited; inspect api.log")
        try:
            if read_json(f"http://127.0.0.1:{PORT}/health")["status"] == "ok":
                break
        except Exception:
            pass
        if time.monotonic() >= deadline:
            raise RuntimeError("API did not start; inspect api.log")
        time.sleep(2)
    readiness = read_json(f"http://127.0.0.1:{PORT}/deployment/readiness", backend_key)
    if not readiness.get("ready"):
        raise RuntimeError("AI readiness failed; inspect API and GPU logs")
    capabilities = readiness.get("style_capabilities", {})
    expected_release = STYLE_RELEASE_EXPECTED
    if (capabilities.get("release_id") != expected_release["release_id"]
            or capabilities.get("manifest_sha256") != expected_release["manifest_sha256"]
            or not set(expected_release["enabled_style_ids"]).issubset(capabilities.get("renderer_styles", []) + capabilities.get("model_styles", {}).get("hairstyle", []))):
        raise RuntimeError("Loaded facade does not match the approved Nail release")
    # Bootstrap already downloaded cloudflared; point only at the authenticated facade.
    binary = WORK / "gpu-session/cloudflared"
    if not binary.exists():
        import shutil
        with urlopen("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", timeout=90) as source, binary.open("wb") as target:
            shutil.copyfileobj(source, target)
        binary.chmod(0o700)
    with (WORK / "tunnel.log").open("w") as log:
        tunnel = subprocess.Popen([str(binary), "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
                                  cwd=WORK, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    deadline = time.monotonic() + 240
    while True:
        if tunnel.poll() is not None:
            raise RuntimeError("API tunnel exited; inspect tunnel.log")
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", (WORK / "tunnel.log").read_text())
        if match:
            try:
                if read_json(match[0] + "/deployment/readiness", backend_key).get("ready"):
                    break
            except Exception:
                pass
        if time.monotonic() >= deadline:
            raise RuntimeError("Public API readiness failed; inspect tunnel.log")
        time.sleep(3)
    record = {"status": "READY_FOR_REHEARSAL", "url": match[0], "api_pid": api.pid,
              "tunnel_pid": tunnel.pid, "readiness": readiness, "host_ram_bytes": memory,
              "full_generation_verified": False}
    (WORK / "demo-session.json").write_text(json.dumps(record, indent=2))
    website = os.environ.get("BEAUTYCORE_PUBLIC_URL", "").strip()
    if website:
        import importlib.util
        registrar_path = WORK / "runtime_registration.py"
        if not registrar_path.is_file():
            raise RuntimeError("Automatic registration helper is missing; use the corrected client notebook")
        spec = importlib.util.spec_from_file_location("demo_registrar", registrar_path)
        registrar = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(registrar)
        record.update(session_id=session_id, started_at=started_at, status="AI_READY_CONNECTION_PENDING")
        (WORK / "demo-session.json").write_text(json.dumps(record, indent=2))
        try:
            ack = registrar.register(website, record, backend_key)
        except registrar.RegistrationError:
            print("AI is running, but website registration was not acknowledged. Inspect registration status; do not rerun GPU startup.", flush=True)
            raise
        record.update(status="CONNECTED_FOR_REHEARSAL", registration_expires_at=ack["expires_at"])
        (WORK / "demo-session.json").write_text(json.dumps(record, indent=2))
        with (WORK / "registration.log").open("w") as log:
            heartbeat = subprocess.Popen([sys.executable, str(registrar_path), website, str(WORK / "demo-session.json")],
                env={**env, "AI_BACKEND_API_KEY": backend_key}, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        record["heartbeat_pid"] = heartbeat.pid
        (WORK / "demo-session.json").write_text(json.dumps(record, indent=2))
        print("CONNECTED TO WEBSITE:", website, flush=True)
        print("No Render URL edit is needed. Keep the Kaggle session running.", flush=True)
    else:
        print("Render AI_FASTAPI_URL:", match[0], flush=True)
    print("READY_FOR_REHEARSAL: run the full application checklist before the defense.", flush=True)



main()


In [ ]:
record = json.loads((CLIENT_WORK/'demo-session.json').read_text())
print('Status:', record['status'])
print('Website:', BEAUTYCORE_PUBLIC_URL)
print('Release:', STYLE_RELEASE_EXPECTED['release_id'])
print('Added styles:', ', '.join(STYLE_RELEASE_EXPECTED['enabled_style_ids']))
print('Connected status still requires the website rehearsal. Keep this session running.')
